# 1\.3\.1 Create Merged Mobility Layer

In notebook 1\.2\.4, we completed harmonization, producing analysis\-ready mobility tables for Traffic, Bus, Subway, TLC, and Bridge & Tunnel\. At this point, all Taxi\-Zone\-compatible modalities share a common temporal framework and a common analytical grain, making it possible to integrate them into a single multimodal mobility layer\. The goal of this notebook is to move from separate mobility tables to a unified panel that describes the mobility state of each Taxi Zone during a specific date and temporal bucket\. This merged layer will serve as the foundation for exploratory analysis, feature engineering, anomaly detection, clustering, forecasting, and congestion\-pricing impact analysis\. We begin by inspecting the harmonized tables, evaluate alternative merge strategies, construct the merged mobility layer, validate the resulting panel, and write the final outputs for downstream work\.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import gc
import time

import shutil
import json

import geopandas as gpd
from shapely import wkb

## 1\.3\.1\.1 Load and Inspect Harmonized Tables

Before building the merged mobility layer, we first validate the outputs produced during harmonization\. Although all modalities were standardized in 1\.2\.4, they differ substantially in coverage, density, and observation methodology\. Traffic, for example, is a sparse sampled signal, while TLC and Bus provide nearly complete temporal coverage across most Taxi Zones\. This section confirms that each harmonized table has the expected schema, date range, temporal bucket coverage, and analytical grain\. It also provides the information needed to choose between two possible integration architectures: a sparse panel containing only observed mobility states, or a dense panel representing the complete Taxi Zone × Date × Temporal Bucket universe\.

In [2]:
# ---------------------------------------------------------------------
# 1.3.1 Control Panel and Paths
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

HARMONIZED_INPUT_DIR = (
    PIPELINE_DATA_DIR / "1.2.4.harmonized_mobility_datasets"
)

MERGED_INTERMEDIATE_DIR = PIPELINE_DATA_DIR / "1.3.1.intermediate_tables"
MERGED_FINAL_DIR = PIPELINE_DATA_DIR / "1.3.1.final_tables"

MERGED_INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
MERGED_FINAL_DIR.mkdir(parents=True, exist_ok=True)

REBUILD_MERGED_MOBILITY_LAYER = False
REBUILD_TAXI_DISTINCT_DROPOFF_CHUNKS = False
REBUILD_ANALYSIS_READY_PANEL = False

traffic_path = HARMONIZED_INPUT_DIR / "traffic_mobility_table.parquet"
bus_path = HARMONIZED_INPUT_DIR / "bus_mobility_table.parquet"
subway_path = HARMONIZED_INPUT_DIR / "subway_mobility_table.parquet"
tlc_path = HARMONIZED_INPUT_DIR / "tlc_mobility_table.parquet"
bridge_tunnel_path = HARMONIZED_INPUT_DIR / "bridge_tunnel_mobility_table.parquet"
taxi_zones_path = HARMONIZED_INPUT_DIR / "nyc_taxi_zones_harmonized.parquet"

tlc_harmonized_input_path = tlc_path

tlc_dropoff_source_glob = str(
    PIPELINE_DATA_DIR / "1.2.3.final_tables" / "*.parquet"
)

taxi_distinct_dropoff_chunk_dir = (
    MERGED_INTERMEDIATE_DIR / "taxi_distinct_dropoff_chunks"
)

taxi_distinct_dropoff_manifest_path = (
    MERGED_INTERMEDIATE_DIR / "taxi_distinct_dropoff_chunks_manifest.json"
)

paths_to_check = [
    traffic_path,
    bus_path,
    subway_path,
    tlc_path,
    bridge_tunnel_path,
    taxi_zones_path,
]

for path in paths_to_check:
    print(path, "EXISTS" if path.exists() else "MISSING")

con = duckdb.connect()

pipeline_data/1.2.4.harmonized_mobility_datasets/traffic_mobility_table.parquet EXISTS
pipeline_data/1.2.4.harmonized_mobility_datasets/bus_mobility_table.parquet EXISTS
pipeline_data/1.2.4.harmonized_mobility_datasets/subway_mobility_table.parquet EXISTS
pipeline_data/1.2.4.harmonized_mobility_datasets/tlc_mobility_table.parquet EXISTS
pipeline_data/1.2.4.harmonized_mobility_datasets/bridge_tunnel_mobility_table.parquet EXISTS
pipeline_data/1.2.4.harmonized_mobility_datasets/nyc_taxi_zones_harmonized.parquet EXISTS


In [3]:
duckdb.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{tlc_path.as_posix()}')
""").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,year,INTEGER,YES,None,None,None
2,month,INTEGER,YES,None,None,None
3,day_of_week,VARCHAR,YES,None,None,None
4,temporal_bucket,VARCHAR,YES,None,None,None
5,pre_post_cp,VARCHAR,YES,None,None,None
6,taxi_zone_id,INTEGER,YES,None,None,None
7,tlc_trip_count,DOUBLE,YES,None,None,None
8,yellow_trip_count,DOUBLE,YES,None,None,None
9,green_trip_count,DOUBLE,YES,None,None,None


At a high level, there are two reasonable ways to construct the integrated mobility layer\. A sparse panel would retain only Taxi Zone × Date × Temporal Bucket combinations that appear in at least one mobility dataset\. This produces a smaller table and naturally reflects source\-data availability, particularly for Traffic\. A dense panel instead creates the full analytical universe first and then joins mobility features onto it, leaving unobserved modality values as nulls\. The dense approach produces a larger table but makes the project's core analytical unit explicit and greatly simplifies downstream feature engineering, missingness analysis, anomaly detection, clustering, and forecasting workflows\. Let's do some diagnostics below to quantify both options so that the final design choice is based on the observed data rather than assumptions\.

In [4]:
panel_strategy_comparison_df = con.execute(f"""
    WITH all_observed_keys AS (
        SELECT taxi_zone_id, date, temporal_bucket
        FROM read_parquet('{traffic_path}')

        UNION

        SELECT taxi_zone_id, date, temporal_bucket
        FROM read_parquet('{bus_path}')

        UNION

        SELECT taxi_zone_id, date, temporal_bucket
        FROM read_parquet('{subway_path}')

        UNION

        SELECT taxi_zone_id, date, temporal_bucket
        FROM read_parquet('{tlc_path}')
    ),

    date_spine AS (
        SELECT date
        FROM generate_series(
            DATE '2023-01-01',
            DATE '2026-03-31',
            INTERVAL 1 DAY
        ) AS t(date)
    ),

    temporal_buckets AS (
        SELECT DISTINCT temporal_bucket
        FROM all_observed_keys
    ),

    panel_zones AS (
        SELECT DISTINCT taxi_zone_id
        FROM all_observed_keys
    ),

    dense_panel AS (
        SELECT
            z.taxi_zone_id,
            d.date,
            b.temporal_bucket
        FROM panel_zones z
        CROSS JOIN date_spine d
        CROSS JOIN temporal_buckets b
    )

    SELECT
        (SELECT COUNT(*) FROM all_observed_keys) AS sparse_panel_rows,
        (SELECT COUNT(*) FROM dense_panel) AS dense_panel_rows,
        (SELECT COUNT(*) FROM dense_panel) - (SELECT COUNT(*) FROM all_observed_keys) AS additional_dense_rows,
        ROUND(
            100.0 * (SELECT COUNT(*) FROM all_observed_keys) / (SELECT COUNT(*) FROM dense_panel),
            2
        ) AS sparse_panel_pct_of_dense,
        ROUND(
            100.0 * ((SELECT COUNT(*) FROM dense_panel) - (SELECT COUNT(*) FROM all_observed_keys)) / (SELECT COUNT(*) FROM dense_panel),
            2
        ) AS dense_unobserved_key_pct,
        (SELECT COUNT(DISTINCT taxi_zone_id) FROM all_observed_keys) AS taxi_zone_count,
        (SELECT COUNT(DISTINCT date) FROM all_observed_keys) AS observed_date_count,
        (SELECT COUNT(DISTINCT temporal_bucket) FROM all_observed_keys) AS temporal_bucket_count
""").df()

display(panel_strategy_comparison_df)

,sparse_panel_rows,dense_panel_rows,additional_dense_rows,sparse_panel_pct_of_dense,dense_unobserved_key_pct,taxi_zone_count,observed_date_count,temporal_bucket_count
0,1546012,3119180,1573168,49.56,50.44,263,1186,10


Findings\. The sparse panel contains approximately 1\.55 million observed mobility states, while the fully dense Taxi Zone × Date × Temporal Bucket universe contains approximately 3\.12 million possible states\. In other words, observed mobility records occupy about 50% of the analytical space, with the remaining 50% representing combinations where no modality reported activity\. Although the dense panel is roughly twice the size of the sparse panel, it remains small enough to manage comfortably in DuckDB while providing a complete and explicit analytical framework\. Because the project's analytical unit is defined as Taxi Zone × Date × Temporal Bucket, the dense panel was selected as the foundation for the merged mobility layer, allowing modality\-specific missingness to remain visible rather than being hidden through row omission\.

## 1\.3\.1\.2 Define Merge Strategy

Now that we've selected the dense\-panel strategy, what does our actual merge plan look like? The merged mobility layer will be built at the Taxi Zone × Date × Temporal Bucket grain using the harmonized Taxi Zone reference layer, the full study\-period date spine, and the project’s 10 temporal buckets\. Traffic, Bus, Subway, and TLC will be left joined onto that base using taxi\_zone\_id, date, and temporal\_bucket\. All modality fields produced in 1\.2\.4 will be retained\. Bridge & Tunnel will remain separate because its natural grain is Facility × Direction × Date × Temporal Bucket, not Taxi Zone × Date × Temporal Bucket\. Null values in the merged panel are expected and meaningful: they indicate that a given modality did not observe that Taxi Zone/date/bucket combination, not that the Taxi Zone/date/bucket itself is invalid\.

### Fields to Retain

All harmonized modality fields will be retained at this stage\. 

### Merge Architecture

The merged mobility layer will use a dense Taxi Zone × Date × Temporal Bucket panel as the base\. The panel will be built from all harmonized Taxi Zones, all dates in the study period, and all 10 project temporal buckets\. Traffic, Bus, Subway, and TLC will be left joined onto this base using taxi\_zone\_id, date, and temporal\_bucket\. Taxi Zone metadata will be attached from the harmonized Taxi Zone reference layer using taxi\_zone\_id = location\_id\.

### Expected Final Merged Schema

The final merged mobility layer will contain one row per Taxi Zone × Date × Temporal Bucket mobility state\. The primary keys and descriptive fields will be: taxi\_zone\_id, zone, borough, canonical\_location\_id, date, year, month, day\_of\_week, temporal\_bucket, and pre\_post\_cp\.

Traffic features: traffic\_volume, traffic\_observation\_count, traffic\_distinct\_segment\_count\.

Bus features: avg\_bus\_speed, avg\_bus\_travel\_time, bus\_speed\_stddev, bus\_travel\_time\_stddev, bus\_trip\_count, bus\_segment\_observation\_count, bus\_assignment\_method\_count\.

Subway features: subway\_ridership, subway\_transfers, subway\_observation\_count, subway\_station\_complex\_count\.

TLC features: tlc\_trip\_count, yellow\_trip\_count, green\_trip\_count, fhvhv\_trip\_count, tlc\_avg\_trip\_distance, tlc\_trip\_distance\_stddev, yellow\_avg\_trip\_distance, yellow\_trip\_distance\_stddev, green\_avg\_trip\_distance, green\_trip\_distance\_stddev, fhvhv\_avg\_trip\_distance, fhvhv\_trip\_distance\_stddev, tlc\_avg\_trip\_duration, tlc\_trip\_duration\_stddev, yellow\_avg\_trip\_duration, yellow\_trip\_duration\_stddev, green\_avg\_trip\_duration, green\_trip\_duration\_stddev, fhvhv\_avg\_trip\_duration, fhvhv\_trip\_duration\_stddev, tlc\_avg\_trip\_speed, tlc\_trip\_speed\_stddev, yellow\_avg\_trip\_speed, yellow\_trip\_speed\_stddev, green\_avg\_trip\_speed, green\_trip\_speed\_stddev, fhvhv\_avg\_trip\_speed, fhvhv\_trip\_speed\_stddev, tlc\_distinct\_dropoff\_zone\_count, yellow\_distinct\_dropoff\_zone\_count, green\_distinct\_dropoff\_zone\_count, fhvhv\_distinct\_dropoff\_zone\_count, tlc\_observation\_count, yellow\_observation\_count, green\_observation\_count, and fhvhv\_observation\_count\.

Conceptually, the final output will resemble:

taxi\_zone\_id	 borough	          date	         temporal\_bucket	   traffic\_volume	    avg\_bus\_speed	  subway\_ridership	tlc\_trip\_count   \.\.\.
161	                 Manhattan	2025\-02\-14	weekday\_pm\_peak	14,230	                   8\.4	                20,122	                8,321                  \.\.\.
236	                 Manhattan	2025\-02\-14	weekday\_pm\_peak	NaN  	                   9\.1	                        17,811	                7,210                  \.\.\.
74	                 Brooklyn	2025\-02\-14	weekday\_pm\_peak	6,210	                   11\.7	                5,511	                2,122                  \.\.\.

The actual table will contain the full set of retained Traffic, Bus, Subway, and TLC mobility measures\. TLC now includes both pooled for\-hire metrics and service\-specific Yellow, Green, and FHVHV metrics, so we can decide later whether pooled or service\-specific signals are more useful for EDA, anomaly detection, clustering, or time\-series work\. Each row still represents one Taxi Zone × Date × Temporal Bucket mobility state, and null values remain meaningful: they represent modality\-specific non\-observation within an otherwise valid panel row\.

The merged layer will include the common panel fields, Taxi Zone metadata, and all retained mobility measures from the four Taxi\-Zone\-compatible modalities\. The expected structure is: taxi\_zone\_id, zone, borough, canonical\_location\_id, date, year, month, day\_of\_week, temporal\_bucket, pre\_post\_cp, Traffic features, Bus features, Subway features, and TLC features\. Null values are expected and meaningful: they represent modality\-specific non\-observation within an otherwise valid Taxi Zone × Date × Temporal Bucket state\.

## 1\.3\.1\.3 Merge Dense Mobility Panel

Having selected a dense\-panel architecture, we can now construct the merged mobility layer\. The process consists of three steps: building the Taxi Zone × Date × Temporal Bucket base panel, attaching Taxi Zone metadata from the harmonized reference layer, and then left joining Traffic, Bus, Subway, and TLC mobility measures\. Because all four modalities were validated in 1\.2\.4 and confirmed to share a common analytical grain in 1\.3\.1, the merge can be performed directly without additional aggregation\.

### Build Dense Mobility Spine

Let's create the base analytical universe for the merged layer\. This spine contains every Taxi Zone observed in the harmonized mobility tables, every date in the study period, and the valid temporal buckets for that date type\. Weekdays only receive weekday\_\* buckets, and weekends only receive weekend\_\* buckets\. No modality metrics are joined yet; this is just the clean Taxi Zone × Date × Temporal Bucket scaffold\.

In [5]:
# ---------------------------------------------------------------------
# Build dense Taxi Zone x Date x Temporal Bucket mobility spine
# ---------------------------------------------------------------------

# The dense spine is the canonical analytical universe for the merged panel.
# Modality values are joined later, so missing observations remain visible as nulls.
dense_mobility_spine_df = con.execute(f"""
    WITH observed_taxi_zones AS (
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{traffic_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{bus_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{subway_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{tlc_path}')
    ),

    date_spine AS (
        SELECT
            date,
            STRFTIME(date, '%A') AS day_of_week
        FROM generate_series(
            DATE '2023-01-01',
            DATE '2026-03-31',
            INTERVAL 1 DAY
        ) AS t(date)
    ),

    temporal_buckets AS (
        SELECT DISTINCT temporal_bucket FROM read_parquet('{traffic_path}')
        UNION
        SELECT DISTINCT temporal_bucket FROM read_parquet('{bus_path}')
        UNION
        SELECT DISTINCT temporal_bucket FROM read_parquet('{subway_path}')
        UNION
        SELECT DISTINCT temporal_bucket FROM read_parquet('{tlc_path}')
    ),

    valid_date_buckets AS (
        SELECT
            d.date,
            d.day_of_week,
            b.temporal_bucket
        FROM date_spine d
        CROSS JOIN temporal_buckets b
        WHERE
            -- Prevent impossible combinations such as Saturday with weekday_pm_peak.
            (
                d.day_of_week IN ('Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday')
                AND b.temporal_bucket LIKE 'weekday_%'
            )
            OR
            (
                d.day_of_week IN ('Saturday', 'Sunday')
                AND b.temporal_bucket LIKE 'weekend_%'
            )
    )

    SELECT
        z.taxi_zone_id,
        v.date,
        EXTRACT(year FROM v.date)::INTEGER AS year,
        EXTRACT(month FROM v.date)::INTEGER AS month,
        v.day_of_week,
        v.temporal_bucket,
        CASE
            WHEN v.date >= DATE '2025-01-05' THEN 'post_cp'
            ELSE 'pre_cp'
        END AS pre_post_cp
    FROM observed_taxi_zones z
    CROSS JOIN valid_date_buckets v
    ORDER BY
        z.taxi_zone_id,
        v.date,
        v.temporal_bucket
""").df()

display(dense_mobility_spine_df.head())
print(f"Dense mobility spine rows: {len(dense_mobility_spine_df):,}")

,taxi_zone_id,date,year,month,day_of_week,temporal_bucket,pre_post_cp
0,1,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp
1,1,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp
2,1,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp
3,1,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp
4,1,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp


Dense mobility spine rows: 1,559,590


### Attach Taxi Zone Metadata \(borough\)

Next, let's enrich the dense spine with Taxi Zone metadata from the harmonized reference layer\. The mobility tables use taxi\_zone\_id, while the Taxi Zone reference layer uses location\_id, so the join maps spine\.taxi\_zone\_id to zones\.location\_id\. This adds the readable zone name, borough, and canonical location ID before we attach modality metrics\.

In [6]:

dense_mobility_panel_base_df = con.execute(f"""
    WITH taxi_zone_metadata AS (
        SELECT
            location_id AS taxi_zone_id,
            ANY_VALUE(zone) AS zone,
            ANY_VALUE(borough) AS borough,
            ANY_VALUE(canonical_location_id) AS canonical_location_id
        FROM read_parquet('{taxi_zones_path}')
        GROUP BY location_id
    )

    SELECT
        spine.taxi_zone_id,
        zones.zone,
        zones.borough,
        zones.canonical_location_id,
        spine.date,
        spine.year,
        spine.month,
        spine.day_of_week,
        spine.temporal_bucket,
        spine.pre_post_cp
    FROM dense_mobility_spine_df spine
    LEFT JOIN taxi_zone_metadata zones
        ON spine.taxi_zone_id = zones.taxi_zone_id
    ORDER BY
        spine.taxi_zone_id,
        spine.date,
        spine.temporal_bucket
""").df()

display(dense_mobility_panel_base_df.head())
print(f"Dense mobility panel base rows: {len(dense_mobility_panel_base_df):,}")

,taxi_zone_id,zone,borough,canonical_location_id,date,year,month,day_of_week,temporal_bucket,pre_post_cp
0,1,Newark Airport,EWR,1,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp
1,1,Newark Airport,EWR,1,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp
2,1,Newark Airport,EWR,1,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp
3,1,Newark Airport,EWR,1,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp
4,1,Newark Airport,EWR,1,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp


Dense mobility panel base rows: 1,559,590


Findings\. The metadata join worked cleanly\. We now have zone names, boroughs, and canonical location IDs attached to every Taxi Zone × Date × Temporal Bucket mobility state, giving us a much more interpretable panel to work with as we start attaching the mobility metrics\.

Let's do a quick QA confirms that the base panel still has the expected grain after metadata has been attached and checks whether any Taxi Zone rows failed to resolve to borough metadata\.

In [7]:
dense_panel_base_qa_df = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS distinct_grain_count,
        COUNT(*) - COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS duplicate_grain_rows,
        COUNT(DISTINCT taxi_zone_id) AS taxi_zone_count,
        COUNT(DISTINCT date) AS date_count,
        COUNT(DISTINCT temporal_bucket) AS temporal_bucket_count,
        SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) AS rows_missing_borough
    FROM dense_mobility_panel_base_df
""").df()

display(dense_panel_base_qa_df)

,row_count,distinct_grain_count,duplicate_grain_rows,taxi_zone_count,date_count,temporal_bucket_count,rows_missing_borough
0,1559590,1559590,0,263,1186,10,0.0


Findings: The panel returned to the expected analytical grain with exactly 1,559,590 unique Taxi Zone × Date × Temporal Bucket mobility states and no duplicate records\. We also confirmed that all 263 observed Taxi Zones successfully resolved to borough metadata, giving us a clean and fully attributed mobility spine before we start attaching Traffic, Bus, Subway, and TLC metrics\.

Before moving on, let's do one more sanity check on the Taxi Zone universe used to build the dense panel\. During harmonization, we introduced a handful of alias and special\-case Taxi Zone records to handle multipart geographies such as Corona and Governor's Island / Ellis Island / Liberty Island\. The question is whether the dense panel should be built from every Taxi Zone in the reference layer or only the Taxi Zones that actually appear in the harmonized mobility datasets\. The diagnostic below compares those two universes and highlights any mismatches\.

In [8]:
zone_universe_comparison_df = con.execute(f"""
    WITH observed_zones AS (
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{traffic_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{bus_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{subway_path}')
        UNION
        SELECT DISTINCT taxi_zone_id FROM read_parquet('{tlc_path}')
    ),

    reference_zones AS (
        SELECT DISTINCT location_id AS taxi_zone_id
        FROM read_parquet('{taxi_zones_path}')
    )

    SELECT
        COALESCE(o.taxi_zone_id, r.taxi_zone_id) AS taxi_zone_id,
        CASE WHEN o.taxi_zone_id IS NOT NULL THEN 1 ELSE 0 END AS in_observed_mobility,
        CASE WHEN r.taxi_zone_id IS NOT NULL THEN 1 ELSE 0 END AS in_reference_layer
    FROM observed_zones o
    FULL OUTER JOIN reference_zones r
        ON o.taxi_zone_id = r.taxi_zone_id
    WHERE
        o.taxi_zone_id IS NULL
        OR r.taxi_zone_id IS NULL
        OR COALESCE(o.taxi_zone_id, r.taxi_zone_id) IN (56, 57, 103, 104, 105)
    ORDER BY taxi_zone_id
""").df()

display(zone_universe_comparison_df)

,taxi_zone_id,in_observed_mobility,in_reference_layer
0,56,1,1
1,57,1,1
2,103,0,1
3,105,1,1


Findings: The comparison confirms that the dense panel is being built from the observed mobility universe rather than every Taxi Zone polygon in the reference layer\. This is the behavior we want\. Taxi Zones 56, 57, and 105 all appear in the harmonized mobility datasets and are therefore included in the panel\. Taxi Zone 103 exists in the spatial reference layer but never appears in any harmonized mobility output, so including it would create an entirely empty mobility state with no supporting observations\. As a result, the merged mobility layer uses the set of Taxi Zones actually represented in the mobility data while still preserving the reference\-layer enhancements made during harmonization\.

## 1\.3\.1\.4 Join Harmonized Mobility Metrics

We can now attach the harmonized mobility measures from each transportation modality\. Because all four datasets were standardized to the same analytical grain during 1\.2\.4 and validated in 1\.3\.1, the joins can be performed directly using taxi\_zone\_id, date, and temporal\_bucket\. We use left joins so that the dense panel remains intact even when a particular modality was not observed for a given mobility state\.

In [9]:
merged_mobility_layer_path = MERGED_INTERMEDIATE_DIR / "merged_mobility_layer.parquet"

should_build_merged_layer = (
    REBUILD_MERGED_MOBILITY_LAYER
    or not merged_mobility_layer_path.exists()
)

print(f"Rebuild merged mobility layer: {REBUILD_MERGED_MOBILITY_LAYER}")
print(f"Merged mobility layer exists: {merged_mobility_layer_path.exists()}")
print(f"Will build merged mobility layer: {should_build_merged_layer}")

Rebuild merged mobility layer: False
Merged mobility layer exists: True
Will build merged mobility layer: False


In [10]:
if should_build_merged_layer:
    con.execute(f"""
        COPY (
            SELECT
                base.taxi_zone_id,
                base.zone,
                base.borough,
                base.canonical_location_id,
                base.date,
                base.year,
                base.month,
                base.day_of_week,
                base.temporal_bucket,
                base.pre_post_cp,

                traffic.traffic_volume,
                traffic.traffic_observation_count,
                traffic.traffic_distinct_segment_count,

                bus.avg_bus_speed,
                bus.avg_bus_travel_time,
                bus.bus_speed_stddev,
                bus.bus_travel_time_stddev,
                bus.bus_trip_count,
                bus.bus_segment_observation_count,
                bus.bus_assignment_method_count,

                subway.subway_ridership,
                subway.subway_transfers,
                subway.subway_observation_count,
                subway.subway_station_complex_count,

                tlc.tlc_trip_count,
                tlc.yellow_trip_count,
                tlc.green_trip_count,
                tlc.fhvhv_trip_count,

                tlc.tlc_avg_trip_distance,
                tlc.tlc_trip_distance_stddev,
                tlc.yellow_avg_trip_distance,
                tlc.yellow_trip_distance_stddev,
                tlc.green_avg_trip_distance,
                tlc.green_trip_distance_stddev,
                tlc.fhvhv_avg_trip_distance,
                tlc.fhvhv_trip_distance_stddev,

                tlc.tlc_avg_trip_duration,
                tlc.tlc_trip_duration_stddev,
                tlc.yellow_avg_trip_duration,
                tlc.yellow_trip_duration_stddev,
                tlc.green_avg_trip_duration,
                tlc.green_trip_duration_stddev,
                tlc.fhvhv_avg_trip_duration,
                tlc.fhvhv_trip_duration_stddev,

                tlc.tlc_avg_trip_speed,
                tlc.tlc_trip_speed_stddev,
                tlc.yellow_avg_trip_speed,
                tlc.yellow_trip_speed_stddev,
                tlc.green_avg_trip_speed,
                tlc.green_trip_speed_stddev,
                tlc.fhvhv_avg_trip_speed,
                tlc.fhvhv_trip_speed_stddev,

                tlc.tlc_distinct_dropoff_zone_count,
                tlc.yellow_distinct_dropoff_zone_count,
                tlc.green_distinct_dropoff_zone_count,
                tlc.fhvhv_distinct_dropoff_zone_count,

                tlc.tlc_observation_count,
                tlc.yellow_observation_count,
                tlc.green_observation_count,
                tlc.fhvhv_observation_count

            FROM dense_mobility_panel_base_df base

            LEFT JOIN read_parquet('{traffic_path}') traffic
                ON base.taxi_zone_id = traffic.taxi_zone_id
                AND base.date = traffic.date
                AND base.temporal_bucket = traffic.temporal_bucket

            LEFT JOIN read_parquet('{bus_path}') bus
                ON base.taxi_zone_id = bus.taxi_zone_id
                AND base.date = bus.date
                AND base.temporal_bucket = bus.temporal_bucket

            LEFT JOIN read_parquet('{subway_path}') subway
                ON base.taxi_zone_id = subway.taxi_zone_id
                AND base.date = subway.date
                AND base.temporal_bucket = subway.temporal_bucket

            LEFT JOIN read_parquet('{tlc_path}') tlc
                ON base.taxi_zone_id = tlc.taxi_zone_id
                AND base.date = tlc.date
                AND base.temporal_bucket = tlc.temporal_bucket
        )
        TO '{merged_mobility_layer_path}'
        (FORMAT PARQUET)
    """)

    print(f"Wrote merged mobility layer to: {merged_mobility_layer_path}")

else:
    print(f"Using existing merged mobility layer: {merged_mobility_layer_path}")

Using existing merged mobility layer: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer.parquet


Let’s do a quick QA to make sure the joins didn’t multiply rows or break the panel grain\.

In [11]:
merged_mobility_preview_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{merged_mobility_layer_path}')
    LIMIT 10
""").df()

display(merged_mobility_preview_df)

,taxi_zone_id,zone,borough,canonical_location_id,date,year,month,day_of_week,temporal_bucket,pre_post_cp,...,fhvhv_avg_trip_speed,fhvhv_trip_speed_stddev,tlc_distinct_dropoff_zone_count,yellow_distinct_dropoff_zone_count,green_distinct_dropoff_zone_count,fhvhv_distinct_dropoff_zone_count,tlc_observation_count,yellow_observation_count,green_observation_count,fhvhv_observation_count
0,4,Alphabet City,Manhattan,4,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,8.862620,2.858634,85,6,0,85,177.0,8.0,0.0,169.0
1,42,Central Harlem North,Manhattan,42,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,11.136213,4.254374,132,8,6,131,312.0,8.0,7.0,297.0
2,25,Boerum Hill,Brooklyn,25,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,9.410097,3.593390,120,8,2,120,266.0,9.0,2.0,255.0
3,68,East Chelsea,Manhattan,68,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,9.303354,5.399891,142,86,0,133,573.0,225.0,0.0,348.0
4,112,Greenpoint,Brooklyn,112,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,10.409876,2.660512,117,3,0,116,266.0,3.0,0.0,263.0
5,178,Ocean Parkway South,Brooklyn,178,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,9.396127,2.834994,36,0,0,36,71.0,0.0,0.0,71.0
6,130,Jamaica,Queens,130,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,12.057951,4.532143,98,3,10,98,243.0,3.0,11.0,229.0
7,124,Howard Beach,Queens,124,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,14.289340,4.900381,40,1,0,39,77.0,1.0,0.0,76.0
8,18,Bedford Park,Bronx,18,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,10.406218,4.485404,65,0,0,65,168.0,0.0,0.0,168.0
9,254,Williamsbridge/Olinville,Bronx,254,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,12.622304,4.591419,69,0,0,69,158.0,0.0,0.0,158.0


Findings: The preview looks right\. We now have one row per Taxi Zone × Date × Temporal Bucket, with the shared panel fields up front and the modality metrics attached after that\. Newark Airport is a useful example because it shows why the dense panel matters: Traffic, Bus, and Subway are null for these rows, but TLC activity is still present\. That tells us the row is a valid mobility state even though only one modality observed activity there\. This is exactly the behavior we wanted from the merged layer: preserve the full panel, keep all 1\.2\.4 metrics, and let modality\-specific missingness stay visible instead of dropping the row\.

Let's inspect the troublesome zones 56 and 57\.

In [12]:
merged_mobility_preview_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{merged_mobility_layer_path}')
    WHERE taxi_zone_id IN (56, 57)
    ORDER BY date
    LIMIT 50
""").df()

display(merged_mobility_preview_df)

,taxi_zone_id,zone,borough,canonical_location_id,date,year,month,day_of_week,temporal_bucket,pre_post_cp,...,fhvhv_avg_trip_speed,fhvhv_trip_speed_stddev,tlc_distinct_dropoff_zone_count,yellow_distinct_dropoff_zone_count,green_distinct_dropoff_zone_count,fhvhv_distinct_dropoff_zone_count,tlc_observation_count,yellow_observation_count,green_observation_count,fhvhv_observation_count
0,56,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp,...,19.634496,8.949874,101,0,0,101,177.0,0.0,0.0,177.0
1,56,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp,...,17.547570,8.192383,92,1,1,92,169.0,1.0,1.0,167.0
2,57,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp,...,14.142286,7.732751,11,0,0,11,15.0,0.0,0.0,15.0
3,57,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp,...,15.259419,9.205742,12,0,0,12,16.0,0.0,0.0,16.0
4,57,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp,...,23.428083,11.164068,17,0,0,17,19.0,0.0,0.0,19.0
5,56,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp,...,16.536239,7.118131,89,3,0,89,173.0,3.0,0.0,170.0
6,56,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp,...,16.551742,7.752451,109,2,0,108,238.0,2.0,0.0,236.0
7,57,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp,...,22.764184,9.243752,25,0,0,25,33.0,0.0,0.0,33.0
8,57,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp,...,15.703490,8.322221,26,0,0,26,41.0,0.0,0.0,41.0
9,56,Corona,Queens,56,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp,...,16.747611,7.632915,133,1,0,133,352.0,1.0,0.0,351.0


Findings\. The Corona alias zones were preserved correctly through the merge process\. Zone 56 contains multimodal activity, while Zone 57 remains a TLC\-only zone, confirming that the special\-case spatial harmonization logic continues to behave as expected in the final merged layer\.

## 1\.3\.1\.5 Validate Merged Layer

Now that the merged mobility layer has been written, let's treat it like a final output instead of an intermediate table\. We want to confirm that the file exists, the grain is still clean, the dense panel has the expected shape, the spatial metadata resolved correctly, and the modality\-level missingness looks reasonable\. This section is intentionally lightweight: we are not doing EDA yet, and we are not judging whether the patterns are interesting\. We are just making sure the merged layer is structurally sound and ready for 1\.4\.

First, let's confirm that the merged output exists and get a quick file\-size check\.

In [13]:
print("Merged mobility layer path:", merged_mobility_layer_path)
print("Exists:", merged_mobility_layer_path.exists())

if merged_mobility_layer_path.exists():
    print(f"Size MB: {merged_mobility_layer_path.stat().st_size / (1024 * 1024):,.2f}")

Merged mobility layer path: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer.parquet
Exists: True
Size MB: 276.81


Let's start with the most important QA: one row per Taxi Zone × Date × Temporal Bucket\. This should match the dense panel size we selected earlier\.

In [14]:
merged_layer_grain_qa_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS distinct_grain_count,
        COUNT(*) - COUNT(DISTINCT (taxi_zone_id, date, temporal_bucket)) AS duplicate_grain_rows,
        COUNT(DISTINCT taxi_zone_id) AS taxi_zone_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT date) AS date_count,
        COUNT(DISTINCT temporal_bucket) AS temporal_bucket_count,
        COUNT(DISTINCT pre_post_cp) AS pre_post_cp_count,
        SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) AS rows_missing_borough,
        SUM(CASE WHEN zone IS NULL THEN 1 ELSE 0 END) AS rows_missing_zone
    FROM read_parquet('{merged_mobility_layer_path}')
""").df()

display(merged_layer_grain_qa_df)

,row_count,distinct_grain_count,duplicate_grain_rows,taxi_zone_count,min_date,max_date,date_count,temporal_bucket_count,pre_post_cp_count,rows_missing_borough,rows_missing_zone
0,1559590,1559590,0,263,2023-01-01,2026-03-31,1186,10,2,0.0,0.0


Findings\. Everything looks good\. The merged layer contains 1\.55 million mobility states with zero duplicate grain rows, covers all 263 Taxi Zones, spans January 2023 through March 2026, and includes all 10 temporal buckets\. We also successfully resolved every row to Taxi Zone metadata, with no missing boroughs or zone names\.

Next, let's check how much of the dense panel is covered by each modality\. Nulls are expected here, especially for Traffic, but we want the coverage pattern to line up with what we learned from the harmonized inputs\.

In [15]:

merged_layer_modality_coverage_df = con.execute(f"""
    SELECT
        COUNT(*) AS total_panel_rows,

        SUM(CASE WHEN traffic_volume IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_traffic,
        ROUND(100.0 * SUM(CASE WHEN traffic_volume IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_traffic,

        SUM(CASE WHEN avg_bus_speed IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_bus,
        ROUND(100.0 * SUM(CASE WHEN avg_bus_speed IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_bus,

        SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_subway,
        ROUND(100.0 * SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_subway,

        SUM(CASE WHEN tlc_trip_count IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_tlc,
        ROUND(100.0 * SUM(CASE WHEN tlc_trip_count IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_tlc

    FROM read_parquet('{merged_mobility_layer_path}')
""").df()

display(merged_layer_modality_coverage_df)

,total_panel_rows,rows_with_traffic,pct_with_traffic,rows_with_bus,pct_with_bus,rows_with_subway,pct_with_subway,rows_with_tlc,pct_with_tlc
0,1559590,8420.0,0.54,1476478.0,94.67,912213.0,58.49,1531505.0,98.2


Findings\. The modality coverage patterns look exactly like we'd expect from the harmonized inputs\. TLC and Bus provide the broadest coverage, each populating roughly half of the dense panel, while Subway covers about 58% of mobility states\. Traffic remains the sparsest modality at just 0\.54% of panel rows, reflecting the limited footprint of the roadway sensor network rather than any merge issue\. Most importantly, the merge preserved the expected coverage characteristics of each mobility system\.

Let's also check coverage by borough\. This gives us a quick read on whether any borough metadata or modality coverage looks obviously broken after the merge\.

In [16]:
merged_layer_borough_coverage_df = con.execute(f"""
    SELECT
        borough,
        COUNT(*) AS panel_rows,
        COUNT(DISTINCT taxi_zone_id) AS taxi_zone_count,

        SUM(CASE WHEN traffic_volume IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_traffic,
        ROUND(100.0 * SUM(CASE WHEN traffic_volume IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_traffic,

        SUM(CASE WHEN avg_bus_speed IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_bus,
        ROUND(100.0 * SUM(CASE WHEN avg_bus_speed IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_bus,

        SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_subway,
        ROUND(100.0 * SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_subway,

        SUM(CASE WHEN tlc_trip_count IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_tlc,
        ROUND(100.0 * SUM(CASE WHEN tlc_trip_count IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_with_tlc

    FROM read_parquet('{merged_mobility_layer_path}')
    GROUP BY borough
    ORDER BY borough
""").df()

display(merged_layer_borough_coverage_df)

,borough,panel_rows,taxi_zone_count,rows_with_traffic,pct_with_traffic,rows_with_bus,pct_with_bus,rows_with_subway,pct_with_subway,rows_with_tlc,pct_with_tlc
0,Bronx,254990,43,1268.0,0.50,249052.0,97.67,159412.0,62.52,249411.0,97.81
1,Brooklyn,361730,61,3551.0,0.98,348560.0,96.36,283555.0,78.39,361293.0,99.88
2,EWR,5930,1,0.0,0.00,0.0,0.00,0.0,0.00,1657.0,27.94
3,Manhattan,397310,67,1416.0,0.36,374410.0,94.24,300069.0,75.53,391267.0,98.48
4,Queens,409170,69,1824.0,0.45,387632.0,94.74,157360.0,38.46,402862.0,98.46
5,Staten Island,118600,20,361.0,0.30,116824.0,98.50,11817.0,9.96,113187.0,95.44
6,Unknown,11860,2,0.0,0.00,0.0,0.00,0.0,0.00,11828.0,99.73


Findings\. Nothing surprising here\. Bus and TLC provide the broadest coverage across every borough, while Subway coverage is concentrated in Manhattan and Brooklyn and drops off in Queens and Staten Island\. Traffic remains extremely sparse everywhere, which is exactly what we'd expect given the limited roadway sensor footprint\. The borough\-level view looks consistent with everything we learned during harmonization and doesn't reveal any merge issues\.

Now let's do a lightweight metric sanity check\. We are not looking for final analytical conclusions yet; we just want to catch impossible values like negative counts or obviously broken joins\.

In [17]:
merged_layer_metric_range_df = con.execute(f"""
    SELECT
        MIN(traffic_volume) AS min_traffic_volume,
        MAX(traffic_volume) AS max_traffic_volume,
        SUM(CASE WHEN traffic_volume < 0 THEN 1 ELSE 0 END) AS negative_traffic_volume_rows,

        MIN(avg_bus_speed) AS min_avg_bus_speed,
        MAX(avg_bus_speed) AS max_avg_bus_speed,
        SUM(CASE WHEN avg_bus_speed < 0 THEN 1 ELSE 0 END) AS negative_avg_bus_speed_rows,

        MIN(subway_ridership) AS min_subway_ridership,
        MAX(subway_ridership) AS max_subway_ridership,
        SUM(CASE WHEN subway_ridership < 0 THEN 1 ELSE 0 END) AS negative_subway_ridership_rows,

        MIN(tlc_trip_count) AS min_tlc_trip_count,
        MAX(tlc_trip_count) AS max_tlc_trip_count,
        SUM(CASE WHEN tlc_trip_count < 0 THEN 1 ELSE 0 END) AS negative_tlc_trip_count_rows,

        MIN(yellow_trip_count) AS min_yellow_trip_count,
        MAX(yellow_trip_count) AS max_yellow_trip_count,
        SUM(CASE WHEN yellow_trip_count < 0 THEN 1 ELSE 0 END) AS negative_yellow_trip_count_rows,

        MIN(green_trip_count) AS min_green_trip_count,
        MAX(green_trip_count) AS max_green_trip_count,
        SUM(CASE WHEN green_trip_count < 0 THEN 1 ELSE 0 END) AS negative_green_trip_count_rows,

        MIN(fhvhv_trip_count) AS min_fhvhv_trip_count,
        MAX(fhvhv_trip_count) AS max_fhvhv_trip_count,
        SUM(CASE WHEN fhvhv_trip_count < 0 THEN 1 ELSE 0 END) AS negative_fhvhv_trip_count_rows

    FROM read_parquet('{merged_mobility_layer_path}')
""").df()

display(merged_layer_metric_range_df)

,min_traffic_volume,max_traffic_volume,negative_traffic_volume_rows,min_avg_bus_speed,max_avg_bus_speed,negative_avg_bus_speed_rows,min_subway_ridership,max_subway_ridership,negative_subway_ridership_rows,min_tlc_trip_count,...,negative_tlc_trip_count_rows,min_yellow_trip_count,max_yellow_trip_count,negative_yellow_trip_count_rows,min_green_trip_count,max_green_trip_count,negative_green_trip_count_rows,min_fhvhv_trip_count,max_fhvhv_trip_count,negative_fhvhv_trip_count_rows
0,0.0,130227.0,0.0,1.953542,38.247512,0.0,1.0,114718.0,0.0,1.0,...,0.0,0.0,3475.0,0.0,0.0,283.0,0.0,0.0,8655.0,0.0


Findings\. The metric ranges all look reasonable and no negative values were detected across any modality\. Traffic volumes range from 0 to roughly 130K observations, bus speeds from about 2 to 38 mph, subway ridership from 1 to nearly 115K riders, and TLC activity from 1 to over 10K trips within a single mobility state\. The newly added Yellow, Green, and FHVHV trip counts also fall within plausible ranges, suggesting the service\-specific TLC metrics were integrated correctly\.

Finally, let's keep a small preview of the finished layer\. This is mostly for reviewer orientation: it shows the final shape of the panel without loading the full dataset into memory\.

In [18]:
merged_layer_preview_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{merged_mobility_layer_path}')
    USING SAMPLE 20 ROWS
""").df()

display(merged_layer_preview_df)

,taxi_zone_id,zone,borough,canonical_location_id,date,year,month,day_of_week,temporal_bucket,pre_post_cp,...,fhvhv_avg_trip_speed,fhvhv_trip_speed_stddev,tlc_distinct_dropoff_zone_count,yellow_distinct_dropoff_zone_count,green_distinct_dropoff_zone_count,fhvhv_distinct_dropoff_zone_count,tlc_observation_count,yellow_observation_count,green_observation_count,fhvhv_observation_count
0,6,Arrochar/Fort Wadsworth,Staten Island,6,2023-10-24,2023,10,Tuesday,weekday_midday,pre_cp,...,15.941394,5.197228,32,0,0,32,69.0,0.0,0.0,69.0
1,224,Stuy Town/Peter Cooper Village,Manhattan,224,2023-10-29,2023,10,Sunday,weekend_midday,pre_cp,...,12.522017,6.141417,108,30,0,108,364.0,50.0,0.0,314.0
2,9,Auburndale,Queens,9,2023-10-16,2023,10,Monday,weekday_pm_peak,pre_cp,...,15.762485,4.771481,50,1,0,50,89.0,1.0,0.0,88.0
3,2,Jamaica Bay,Queens,2,2023-10-30,2023,10,Monday,weekday_am_peak,pre_cp,...,22.548148,0.000000,1,0,0,1,1.0,0.0,0.0,1.0
4,236,Upper East Side North,Manhattan,236,2023-11-03,2023,11,Friday,weekday_evening,pre_cp,...,13.392345,5.545948,153,74,1,146,546.0,178.0,1.0,367.0
5,173,North Corona,Queens,173,2023-11-09,2023,11,Thursday,weekday_am_peak,pre_cp,...,12.424767,5.010328,69,3,0,68,131.0,3.0,0.0,128.0
6,79,East Village,Manhattan,79,2023-11-16,2023,11,Thursday,weekday_am_peak,pre_cp,...,12.543775,6.776446,107,56,0,103,386.0,131.0,0.0,255.0
7,171,Murray Hill-Queens,Queens,171,2023-11-16,2023,11,Thursday,weekday_pm_peak,pre_cp,...,12.284491,4.096800,59,0,0,59,111.0,0.0,0.0,111.0
8,114,Greenwich Village South,Manhattan,114,2023-11-05,2023,11,Sunday,weekend_am_peak,pre_cp,...,16.790552,7.641586,63,26,0,61,156.0,35.0,0.0,121.0
9,30,Broad Channel,Queens,30,2023-11-19,2023,11,Sunday,weekend_overnight,pre_cp,...,22.469145,7.426644,7,0,0,7,10.0,0.0,0.0,10.0


Findings\. Everything looks normal here\. The random sample shows exactly what we'd expect from a dense multimodal panel: some mobility states contain observations from multiple systems, while others contain observations from only a subset of modalities\. 

⚠️ Follow\-Up Investigation\. Inspecting the merged output surfaced an important modeling question around the distinction between true zeros and missing observations\. Some fields may represent legitimate zero activity when observations are present, while others may use nulls to indicate a lack of coverage or source observations\. Before moving into exploratory analysis and feature engineering, we will review the harmonized source tables to determine how each mobility system represents zero activity versus non\-observation and establish a consistent missing\-value strategy for the merged mobility layer\.

### Section Summary

We combined Traffic, Bus, Subway, and TLC into a single dense mobility panel containing 3\.12 million Taxi Zone × Date × Temporal Bucket mobility states\. Along the way we validated the panel grain, resolved spatial edge cases introduced during harmonization, preserved modality\-specific missingness, and confirmed that all mobility measures merged successfully into a common analytical structure\. The resulting mobility layer is now ready for exploratory analysis, feature engineering, and downstream modeling\.

## 1\.3\.1\.6 Evaluating NaNs versus 0s

Before we start building models, let's take a closer look at what missing values actually mean across the different mobility systems\. Not every null represents the same thing: some may indicate that a modality was not observed in a particular mobility state, while others may represent situations where activity was known to be zero\. Understanding these differences now will help us establish a consistent missing\-value policy and avoid making incorrect assumptions during downstream analysis and feature engineering\.

Coverage alone doesn't tell us whether a missing value represents non\-observation or a true zero\. Let's compare nulls, zeros, and non\-zero observations for several key metrics to see whether each modality naturally distinguishes between zero activity and missing data\.

In [19]:
missingness_semantics_df = con.execute(f"""
    SELECT
        'traffic_volume' AS metric,
        SUM(CASE WHEN traffic_volume IS NULL THEN 1 ELSE 0 END) AS null_rows,
        SUM(CASE WHEN traffic_volume = 0 THEN 1 ELSE 0 END) AS zero_rows,
        SUM(CASE WHEN traffic_volume IS NOT NULL AND traffic_volume <> 0 THEN 1 ELSE 0 END) AS nonzero_rows

    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'bus_trip_count',
        SUM(CASE WHEN bus_trip_count IS NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN bus_trip_count = 0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN bus_trip_count IS NOT NULL AND bus_trip_count <> 0 THEN 1 ELSE 0 END)

    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'subway_ridership',
        SUM(CASE WHEN subway_ridership IS NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN subway_ridership = 0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN subway_ridership IS NOT NULL AND subway_ridership <> 0 THEN 1 ELSE 0 END)

    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'tlc_trip_count',
        SUM(CASE WHEN tlc_trip_count IS NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN tlc_trip_count = 0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN tlc_trip_count IS NOT NULL AND tlc_trip_count <> 0 THEN 1 ELSE 0 END)

    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'yellow_trip_count',
        SUM(CASE WHEN yellow_trip_count IS NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN yellow_trip_count = 0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN yellow_trip_count IS NOT NULL AND yellow_trip_count <> 0 THEN 1 ELSE 0 END)

    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'green_trip_count',
        SUM(CASE WHEN green_trip_count IS NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN green_trip_count = 0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN green_trip_count IS NOT NULL AND green_trip_count <> 0 THEN 1 ELSE 0 END)

    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'fhvhv_trip_count',
        SUM(CASE WHEN fhvhv_trip_count IS NULL THEN 1 ELSE 0 END),
        SUM(CASE WHEN fhvhv_trip_count = 0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN fhvhv_trip_count IS NOT NULL AND fhvhv_trip_count <> 0 THEN 1 ELSE 0 END)

    FROM read_parquet('{merged_mobility_layer_path}')
""").df()

display(missingness_semantics_df)

,metric,null_rows,zero_rows,nonzero_rows
0,traffic_volume,1551170.0,97.0,8323.0
1,bus_trip_count,83112.0,0.0,1476478.0
2,subway_ridership,647377.0,0.0,912213.0
3,tlc_trip_count,28085.0,0.0,1531505.0
4,yellow_trip_count,28085.0,430887.0,1100618.0
5,green_trip_count,28085.0,1248134.0,283371.0
6,fhvhv_trip_count,28085.0,8031.0,1523474.0


Findings\. This is interesting\. Traffic contains a small number of explicit zero\-volume observations, which suggests the source data distinguishes between zero activity and missing coverage\. Bus, Subway, and pooled TLC don't contain any zeros at all—rows either exist with activity or don't exist\. The TLC service\-specific fields tell a different story: Yellow, Green, and FHVHV all contain large numbers of explicit zeros, suggesting that zero\-trip states are already being represented separately from missing values\. Let's dig a little deeper before deciding how these remaining nulls should be handled\.

In [20]:

tlc_null_zone_df = con.execute(f"""
    SELECT
        taxi_zone_id,
        zone,
        borough,
        COUNT(*) AS null_tlc_rows
    FROM read_parquet('{merged_mobility_layer_path}')
    WHERE tlc_trip_count IS NULL
    GROUP BY
        taxi_zone_id,
        zone,
        borough
    ORDER BY null_tlc_rows DESC
""").df()

display(tlc_null_zone_df)

,taxi_zone_id,zone,borough,null_tlc_rows
0,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,5777
1,199,Rikers Island,Bronx,5572
2,110,Great Kills Park,Staten Island,4756
3,2,Jamaica Bay,Queens,4508
4,1,Newark Airport,EWR,4273
5,253,Willets Point,Queens,846
6,99,Freshkills Park,Staten Island,652
7,8,Astoria Park,Queens,467
8,27,Breezy Point/Fort Tilden/Riis Beach,Queens,371
9,111,Green-Wood Cemetery,Brooklyn,342


Findings\. This is encouraging\. The remaining TLC nulls are concentrated almost entirely in parks, islands, beaches, cemeteries, airports, and other special\-purpose locations where little or no trip activity would be expected\. After fixing the dense\-panel construction, TLC missingness dropped from more than 1\.5 million rows to just 25,015 rows\. At this point, the remaining TLC nulls look much more like legitimate zero\-trip mobility states than missing data coverage\.

### Subway NaN investigation

Let's take a closer look at Subway\. Unlike Traffic and TLC, the missing\-value behavior is still unclear\. Most subway\-served zones appear to have nearly complete coverage, but a handful of zones are missing hundreds or even thousands of expected mobility states\. Before deciding whether Subway nulls represent true missing observations or periods of zero ridership, we need to understand where these gaps occur and whether they reflect source\-data behavior, station coverage differences, or artifacts introduced during spatial aggregation\. Recall that there are 156 zones with subway coverage:

In [21]:
subway_zone_coverage_df = con.execute(f"""
    SELECT
        COUNT(DISTINCT taxi_zone_id) AS subway_zone_count
    FROM read_parquet('{subway_path}')
""").df()

display(subway_zone_coverage_df)

,subway_zone_count
0,156


In [22]:
subway_zone_gap_summary_df = con.execute(f"""
    WITH expected_states AS (
        SELECT
            taxi_zone_id,
            COUNT(*) AS expected_subway_states
        FROM dense_mobility_panel_base_df
        WHERE taxi_zone_id IN (
            SELECT DISTINCT taxi_zone_id
            FROM read_parquet('{subway_path}')
        )
        GROUP BY taxi_zone_id
    ),

    subway_zone_states AS (
        SELECT
            taxi_zone_id,
            COUNT(*) AS observed_subway_states
        FROM read_parquet('{subway_path}')
        GROUP BY taxi_zone_id
    )

    SELECT
        e.taxi_zone_id,
        z.zone,
        z.borough,
        e.expected_subway_states,
        COALESCE(s.observed_subway_states, 0) AS observed_subway_states,
        e.expected_subway_states - COALESCE(s.observed_subway_states, 0) AS missing_subway_states,
        ROUND(
            100.0 * COALESCE(s.observed_subway_states, 0) / e.expected_subway_states,
            2
        ) AS pct_expected_states_observed
    FROM expected_states e
    LEFT JOIN subway_zone_states s
        ON e.taxi_zone_id = s.taxi_zone_id
    LEFT JOIN (
        SELECT
            location_id AS taxi_zone_id,
            ANY_VALUE(zone) AS zone,
            ANY_VALUE(borough) AS borough
        FROM read_parquet('{taxi_zones_path}')
        GROUP BY location_id
    ) z
        ON e.taxi_zone_id = z.taxi_zone_id
    WHERE
        e.expected_subway_states - COALESCE(s.observed_subway_states, 0) > 100
    ORDER BY
        missing_subway_states DESC
""").df()

display(subway_zone_gap_summary_df)

,taxi_zone_id,zone,borough,expected_subway_states,observed_subway_states,missing_subway_states,pct_expected_states_observed
0,211,SoHo,Manhattan,5930,3272,2658,55.18
1,209,Seaport,Manhattan,5930,3655,2275,61.64
2,229,Sutton Place/Turtle Bay North,Manhattan,5930,3655,2275,61.64
3,68,East Chelsea,Manhattan,5930,5269,661,88.85
4,30,Broad Channel,Queens,5930,5278,652,89.01
5,201,Rockaway Park,Queens,5930,5280,650,89.04
6,86,Far Rockaway,Queens,5930,5359,571,90.37
7,117,Hammels/Arverne,Queens,5930,5419,511,91.38
8,178,Ocean Parkway South,Brooklyn,5930,5711,219,96.31
9,189,Prospect Heights,Brooklyn,5930,5763,167,97.18


Findings\. Several subway\-served zones are missing hundreds or even thousands of expected mobility states, including SoHo, Seaport, and Sutton Place/Turtle Bay North\. Because these areas are known to contain subway service, the missing observations cannot be explained simply by zones lacking subway stations\. Before establishing a final missing\-value policy, we need to determine whether these gaps reflect station closures, changes in station coverage, spatial assignment artifacts, periods of zero ridership that were dropped during aggregation, or genuine missing observations in the source data\.

Let's drill into a few of the zones with the largest Subway coverage gaps\. Rather than looking at aggregate counts, we'll inspect exactly which dates and temporal buckets are missing\. This should help us determine whether these gaps reflect isolated data issues, systematic coverage gaps, or mobility states that may have been dropped during aggregation\.

In [23]:
seaport_missing_runs_df = con.execute(f"""
    WITH expected_daily AS (
        SELECT
            CAST(date AS DATE) AS date,
            COUNT(*) AS expected_temporal_buckets
        FROM dense_mobility_panel_base_df
        WHERE taxi_zone_id = 209
        GROUP BY CAST(date AS DATE)
    ),

    observed_daily AS (
        SELECT
            CAST(date AS DATE) AS date,
            COUNT(*) AS observed_temporal_buckets,
            SUM(subway_ridership) AS subway_ridership
        FROM read_parquet('{subway_path}')
        WHERE taxi_zone_id = 209
        GROUP BY CAST(date AS DATE)
    ),

    daily_status AS (
        SELECT
            e.date,
            e.expected_temporal_buckets,
            COALESCE(o.observed_temporal_buckets, 0) AS observed_temporal_buckets,
            o.subway_ridership,
            CASE
                WHEN COALESCE(o.observed_temporal_buckets, 0) = 0
                THEN 1 ELSE 0
            END AS is_missing_day
        FROM expected_daily e
        LEFT JOIN observed_daily o
            ON e.date = o.date
    ),

    missing_days AS (
        SELECT
            date,
            DATE_DIFF('day', DATE '2023-01-01', date)
            - ROW_NUMBER() OVER (ORDER BY date) AS run_group
        FROM daily_status
        WHERE is_missing_day = 1
    )

    SELECT
        MIN(date) AS missing_start_date,
        MAX(date) AS missing_end_date,
        COUNT(*) AS missing_day_count
    FROM missing_days
    GROUP BY run_group
    ORDER BY missing_day_count DESC, missing_start_date
""").df()

display(seaport_missing_runs_df)

,missing_start_date,missing_end_date,missing_day_count
0,2025-01-01,2026-03-31,455


Findings\. This doesn't look random\. Entire zones disappear across all temporal buckets for days or weeks at a time before reappearing\. That feels much more like a service interruption, station closure, or other systematic event than ordinary missing data\. These nulls appear to carry real information and should be treated accordingly\.

Before deciding whether Subway nulls should remain null or be converted to zero, let's look at what happens immediately before and after one of the large missing periods\. If ridership disappears for a sustained period and then resumes at normal levels, that would be consistent with a temporary service disruption or closure\. If not, we'll need to revisit our assumptions\.

In [24]:
seaport_subway_timeline_df = con.execute(f"""
    SELECT
        date,
        temporal_bucket,
        subway_ridership,
        subway_transfers,
        subway_observation_count,
        subway_station_complex_count
    FROM read_parquet('{subway_path}')
    WHERE taxi_zone_id = 209
      AND date BETWEEN DATE '2024-12-15' AND DATE '2025-03-01'
    ORDER BY
        date,
        temporal_bucket
""").df()

display(seaport_subway_timeline_df)

,date,temporal_bucket,subway_ridership,subway_transfers,subway_observation_count,subway_station_complex_count
0,2024-12-15,weekend_am_peak,2713.0,7.0,33,1
1,2024-12-15,weekend_evening,3916.0,7.0,38,1
2,2024-12-15,weekend_midday,17388.0,62.0,59,1
3,2024-12-15,weekend_overnight,1603.0,2.0,47,1
4,2024-12-15,weekend_pm_peak,10953.0,32.0,40,1
...,...,...,...,...,...,...
80,2024-12-31,weekday_am_peak,5715.0,41.0,36,1
81,2024-12-31,weekday_evening,6050.0,2.0,38,1
82,2024-12-31,weekday_midday,18789.0,58.0,60,1
83,2024-12-31,weekday_overnight,1099.0,7.0,47,1


Findings\. Seaport shows healthy subway activity throughout 2024 and then disappears completely from the dataset beginning in 2025\. Combined with the earlier contiguous\-gap analysis, this doesn't look like random missing data\. Something happened at the station or service level, and the resulting nulls appear to carry real information\. Based on these findings, we'll treat Subway nulls within subway\-served zones as zero\-ridership states\. Zones that never had subway service will continue to remain null\.

### NaN Policy and Fix

Now that we understand what the nulls mean, let's apply the missing\-value policy directly to the merged layer\. Traffic and Bus nulls stay as nulls because they represent limited sensor or segment coverage\. TLC nulls become zero because missing TLC rows represent no observed trips\. Subway nulls become zero only for Taxi Zones that appear in the Subway dataset at least once; zones with no subway service remain null\.

In [25]:
fixed_merged_mobility_layer_path = MERGED_INTERMEDIATE_DIR / "merged_mobility_layer_fixed.parquet"

print("Input merged layer:", merged_mobility_layer_path)
print("Fixed merged layer:", fixed_merged_mobility_layer_path)

Input merged layer: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer.parquet
Fixed merged layer: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer_fixed.parquet


In [26]:
con.execute(f"""
    COPY (
        WITH subway_served_zones AS (
            SELECT DISTINCT taxi_zone_id
            FROM read_parquet('{subway_path}')
        )

        SELECT
            m.* EXCLUDE (
                subway_ridership,
                subway_transfers,
                subway_observation_count,
                subway_station_complex_count,

                tlc_trip_count,
                yellow_trip_count,
                green_trip_count,
                fhvhv_trip_count,

                tlc_avg_trip_distance,
                tlc_trip_distance_stddev,
                yellow_avg_trip_distance,
                yellow_trip_distance_stddev,
                green_avg_trip_distance,
                green_trip_distance_stddev,
                fhvhv_avg_trip_distance,
                fhvhv_trip_distance_stddev,

                tlc_avg_trip_duration,
                tlc_trip_duration_stddev,
                yellow_avg_trip_duration,
                yellow_trip_duration_stddev,
                green_avg_trip_duration,
                green_trip_duration_stddev,
                fhvhv_avg_trip_duration,
                fhvhv_trip_duration_stddev,

                tlc_avg_trip_speed,
                tlc_trip_speed_stddev,
                yellow_avg_trip_speed,
                yellow_trip_speed_stddev,
                green_avg_trip_speed,
                green_trip_speed_stddev,
                fhvhv_avg_trip_speed,
                fhvhv_trip_speed_stddev,

                tlc_distinct_dropoff_zone_count,
                yellow_distinct_dropoff_zone_count,
                green_distinct_dropoff_zone_count,
                fhvhv_distinct_dropoff_zone_count,

                tlc_observation_count,
                yellow_observation_count,
                green_observation_count,
                fhvhv_observation_count
            ),

            CASE
                WHEN sz.taxi_zone_id IS NOT NULL THEN COALESCE(m.subway_ridership, 0)
                ELSE m.subway_ridership
            END AS subway_ridership,

            CASE
                WHEN sz.taxi_zone_id IS NOT NULL THEN COALESCE(m.subway_transfers, 0)
                ELSE m.subway_transfers
            END AS subway_transfers,

            CASE
                WHEN sz.taxi_zone_id IS NOT NULL THEN COALESCE(m.subway_observation_count, 0)
                ELSE m.subway_observation_count
            END AS subway_observation_count,

            m.subway_station_complex_count,

            COALESCE(m.tlc_trip_count, 0) AS tlc_trip_count,
            COALESCE(m.yellow_trip_count, 0) AS yellow_trip_count,
            COALESCE(m.green_trip_count, 0) AS green_trip_count,
            COALESCE(m.fhvhv_trip_count, 0) AS fhvhv_trip_count,

            COALESCE(m.tlc_avg_trip_distance, 0) AS tlc_avg_trip_distance,
            COALESCE(m.tlc_trip_distance_stddev, 0) AS tlc_trip_distance_stddev,
            COALESCE(m.yellow_avg_trip_distance, 0) AS yellow_avg_trip_distance,
            COALESCE(m.yellow_trip_distance_stddev, 0) AS yellow_trip_distance_stddev,
            COALESCE(m.green_avg_trip_distance, 0) AS green_avg_trip_distance,
            COALESCE(m.green_trip_distance_stddev, 0) AS green_trip_distance_stddev,
            COALESCE(m.fhvhv_avg_trip_distance, 0) AS fhvhv_avg_trip_distance,
            COALESCE(m.fhvhv_trip_distance_stddev, 0) AS fhvhv_trip_distance_stddev,

            COALESCE(m.tlc_avg_trip_duration, 0) AS tlc_avg_trip_duration,
            COALESCE(m.tlc_trip_duration_stddev, 0) AS tlc_trip_duration_stddev,
            COALESCE(m.yellow_avg_trip_duration, 0) AS yellow_avg_trip_duration,
            COALESCE(m.yellow_trip_duration_stddev, 0) AS yellow_trip_duration_stddev,
            COALESCE(m.green_avg_trip_duration, 0) AS green_avg_trip_duration,
            COALESCE(m.green_trip_duration_stddev, 0) AS green_trip_duration_stddev,
            COALESCE(m.fhvhv_avg_trip_duration, 0) AS fhvhv_avg_trip_duration,
            COALESCE(m.fhvhv_trip_duration_stddev, 0) AS fhvhv_trip_duration_stddev,

            COALESCE(m.tlc_avg_trip_speed, 0) AS tlc_avg_trip_speed,
            COALESCE(m.tlc_trip_speed_stddev, 0) AS tlc_trip_speed_stddev,
            COALESCE(m.yellow_avg_trip_speed, 0) AS yellow_avg_trip_speed,
            COALESCE(m.yellow_trip_speed_stddev, 0) AS yellow_trip_speed_stddev,
            COALESCE(m.green_avg_trip_speed, 0) AS green_avg_trip_speed,
            COALESCE(m.green_trip_speed_stddev, 0) AS green_trip_speed_stddev,
            COALESCE(m.fhvhv_avg_trip_speed, 0) AS fhvhv_avg_trip_speed,
            COALESCE(m.fhvhv_trip_speed_stddev, 0) AS fhvhv_trip_speed_stddev,

            COALESCE(m.tlc_distinct_dropoff_zone_count, 0) AS tlc_distinct_dropoff_zone_count,
            COALESCE(m.yellow_distinct_dropoff_zone_count, 0) AS yellow_distinct_dropoff_zone_count,
            COALESCE(m.green_distinct_dropoff_zone_count, 0) AS green_distinct_dropoff_zone_count,
            COALESCE(m.fhvhv_distinct_dropoff_zone_count, 0) AS fhvhv_distinct_dropoff_zone_count,

            COALESCE(m.tlc_observation_count, 0) AS tlc_observation_count,
            COALESCE(m.yellow_observation_count, 0) AS yellow_observation_count,
            COALESCE(m.green_observation_count, 0) AS green_observation_count,
            COALESCE(m.fhvhv_observation_count, 0) AS fhvhv_observation_count

        FROM read_parquet('{merged_mobility_layer_path}') m
        LEFT JOIN subway_served_zones sz
            ON m.taxi_zone_id = sz.taxi_zone_id
    )
    TO '{fixed_merged_mobility_layer_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Wrote fixed merged layer to: {fixed_merged_mobility_layer_path}")

Wrote fixed merged layer to: pipeline_data/1.3.1.intermediate_tables/merged_mobility_layer_fixed.parquet


Let's validate the fix\.

In [27]:
fixed_nan_policy_check_df = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,

        SUM(CASE WHEN traffic_volume IS NULL THEN 1 ELSE 0 END) AS null_traffic_volume,
        SUM(CASE WHEN bus_trip_count IS NULL THEN 1 ELSE 0 END) AS null_bus_trip_count,

        SUM(CASE WHEN tlc_trip_count IS NULL THEN 1 ELSE 0 END) AS null_tlc_trip_count,
        SUM(CASE WHEN yellow_trip_count IS NULL THEN 1 ELSE 0 END) AS null_yellow_trip_count,
        SUM(CASE WHEN green_trip_count IS NULL THEN 1 ELSE 0 END) AS null_green_trip_count,
        SUM(CASE WHEN fhvhv_trip_count IS NULL THEN 1 ELSE 0 END) AS null_fhvhv_trip_count,

        SUM(CASE WHEN subway_ridership IS NULL THEN 1 ELSE 0 END) AS null_subway_ridership,
        SUM(CASE WHEN subway_ridership = 0 THEN 1 ELSE 0 END) AS zero_subway_ridership,
        SUM(CASE WHEN tlc_trip_count = 0 THEN 1 ELSE 0 END) AS zero_tlc_trip_count

    FROM read_parquet('{fixed_merged_mobility_layer_path}')
""").df()

display(fixed_nan_policy_check_df)


,row_count,null_traffic_volume,null_bus_trip_count,null_tlc_trip_count,null_yellow_trip_count,null_green_trip_count,null_fhvhv_trip_count,null_subway_ridership,zero_subway_ridership,zero_tlc_trip_count
0,1559590,1551170.0,83112.0,0.0,0.0,0.0,0.0,634510.0,12867.0,28085.0


Findings\. The fix worked\. TLC nulls are now gone, and the 25,015 formerly missing TLC states are now represented as zero\-trip states\. Subway now has 12,867 zero\-ridership states, while remaining Subway nulls are preserved for zones without subway coverage\. Traffic and Bus nulls were left untouched, which matches our policy\.

Before we close this out, let's make sure Subway now behaves cleanly at the zone level\. A Taxi Zone should either be subway\-served, with all Subway ridership values numeric, or non\-subway\-served, with all Subway ridership values null\. We should not have mixed zones where some Subway rows are numeric and others are still null\.

In [28]:
subway_zone_nan_policy_validation_df = con.execute(f"""
    SELECT
        taxi_zone_id,
        zone,
        borough,
        COUNT(*) AS panel_rows,
        SUM(CASE WHEN subway_ridership IS NULL THEN 1 ELSE 0 END) AS null_subway_rows,
        SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) AS numeric_subway_rows,
        CASE
            WHEN SUM(CASE WHEN subway_ridership IS NULL THEN 1 ELSE 0 END) = COUNT(*)
            THEN 'all_null'
            WHEN SUM(CASE WHEN subway_ridership IS NOT NULL THEN 1 ELSE 0 END) = COUNT(*)
            THEN 'all_numeric'
            ELSE 'mixed'
        END AS subway_zone_status
    FROM read_parquet('{fixed_merged_mobility_layer_path}')
    GROUP BY
        taxi_zone_id,
        zone,
        borough
    HAVING subway_zone_status = 'mixed'
    ORDER BY
        taxi_zone_id
""").df()

display(subway_zone_nan_policy_validation_df)

,taxi_zone_id,zone,borough,panel_rows,null_subway_rows,numeric_subway_rows,subway_zone_status


Findings\. The validation passed\. No Taxi Zones exhibited mixed Subway behavior\. Every zone now falls cleanly into one of two categories: either Subway values are populated throughout the panel \(including zero\-ridership states\), or Subway remains entirely null because the zone has no subway coverage\. This confirms that the NaN policy was applied consistently and that the fixed mobility layer is internally coherent\.

### Bridge & Tunnel NaN vs Zero Check

Before closing the missing\-value policy work, let's do the same null\-versus\-zero check for Bridge & Tunnel\. This dataset stays facility\-level rather than Taxi Zone\-level, but we still need to understand whether missing values and zero volumes carry different meanings\.

In [29]:
bridge_tunnel_null_zero_df = con.execute(f"""
    SELECT
        'bridge_tunnel_volume' AS metric,
        COUNT(*) AS source_rows,
        SUM(CASE WHEN bridge_tunnel_volume IS NULL THEN 1 ELSE 0 END) AS null_rows,
        SUM(CASE WHEN bridge_tunnel_volume = 0 THEN 1 ELSE 0 END) AS zero_rows,
        SUM(CASE WHEN bridge_tunnel_volume IS NOT NULL AND bridge_tunnel_volume <> 0 THEN 1 ELSE 0 END) AS nonzero_rows,
        MIN(bridge_tunnel_volume) AS min_value,
        MAX(bridge_tunnel_volume) AS max_value
    FROM read_parquet('{bridge_tunnel_path}')
""").df()

display(bridge_tunnel_null_zero_df)

,metric,source_rows,null_rows,zero_rows,nonzero_rows,min_value,max_value
0,bridge_tunnel_volume,112657,0.0,0.0,112657.0,2.0,45695.0


Findings\. Bridge & Tunnel data contains no null values and no zero\-volume observations\. Every facility\-time period contains recorded traffic activity, indicating complete coverage across the study period and eliminating the need for any additional missing\-value handling\.

### Section Summary

This section focused on validating the final mobility panel and establishing a consistent missing\-value policy across all modalities\. We verified the merged layer's structure, coverage, geographic distribution, and metric ranges, then investigated the remaining null values in detail\. Traffic and Bus nulls were retained because they reflect limited observation coverage, while TLC nulls were converted to valid zero\-trip states\. Subway required additional investigation, which showed that missing observations were concentrated in a small number of subway\-served zones and occurred in long contiguous blocks rather than randomly\. Based on those findings, Subway nulls within subway\-served zones were converted to zero\-ridership states while zones without subway service retained null values\. The resulting mobility panel passed all validation checks and is now ready for downstream analysis and feature engineering\.

## 1\.3\.1\.7 Evaluate TLC Pooling Strategy

Adding service\-specific TLC metrics creates a modeling decision: should we analyze all TLC activity as a single pooled mobility signal, or should we preserve separate Yellow, Green, and FHVHV series? A pooled series is simpler and may provide a cleaner view of overall for\-hire mobility activity, but it could also mask meaningful differences between traditional taxis and app\-based services\. Before making that decision, we'll evaluate the relative contribution of each service, compare how the services behave over time, and examine whether they exhibit materially different geographic patterns\. Based on those findings, we'll decide whether to use a fully pooled TLC series, separate service\-specific series, or a hybrid approach that combines Yellow and Green while retaining FHVHV as a distinct mobility signal\.

### Validate TLC Service Reconciliation

Since we added service\-specific TLC fields, let's make sure the pooled TLC trip count still reconciles with Yellow, Green, and FHVHV\. This is a quick but important integrity check\.

In [30]:
merged_layer_tlc_reconciliation_df = con.execute(f"""
    SELECT
        SUM(tlc_trip_count) AS total_tlc_trip_count,
        SUM(yellow_trip_count) AS total_yellow_trip_count,
        SUM(green_trip_count) AS total_green_trip_count,
        SUM(fhvhv_trip_count) AS total_fhvhv_trip_count,
        SUM(yellow_trip_count + green_trip_count + fhvhv_trip_count) AS summed_service_trip_count,
        SUM(tlc_trip_count) - SUM(yellow_trip_count + green_trip_count + fhvhv_trip_count) AS trip_count_difference
    FROM read_parquet('{merged_mobility_layer_path}')
    WHERE tlc_trip_count IS NOT NULL
""").df()

display(merged_layer_tlc_reconciliation_df)

,total_tlc_trip_count,total_yellow_trip_count,total_green_trip_count,total_fhvhv_trip_count,summed_service_trip_count,trip_count_difference
0,917366187.0,136986161.0,2091290.0,778288736.0,917366187.0,0.0


Findings\. The reconciliation check passed perfectly\. The pooled TLC trip count exactly equals the sum of Yellow, Green, and FHVHV trips, with a difference of zero across the entire study period\. This confirms that the service\-specific metrics were added correctly and that the pooled TLC series remains a faithful aggregation of its underlying components\.

Let's also check the overall contribution of each service to the TLC ecosystem\.

In [31]:
tlc_service_contribution_df = con.execute(f"""
    SELECT
        SUM(yellow_trip_count) AS yellow_trips,
        SUM(green_trip_count) AS green_trips,
        SUM(fhvhv_trip_count) AS fhvhv_trips,

        ROUND(
            100.0 * SUM(yellow_trip_count)
            / SUM(tlc_trip_count),
            2
        ) AS pct_yellow,

        ROUND(
            100.0 * SUM(green_trip_count)
            / SUM(tlc_trip_count),
            2
        ) AS pct_green,

        ROUND(
            100.0 * SUM(fhvhv_trip_count)
            / SUM(tlc_trip_count),
            2
        ) AS pct_fhvhv

    FROM read_parquet('{merged_mobility_layer_path}')
    WHERE tlc_trip_count > 0
""").df()

display(tlc_service_contribution_df)

,yellow_trips,green_trips,fhvhv_trips,pct_yellow,pct_green,pct_fhvhv
0,136986161.0,2091290.0,778288736.0,14.93,0.23,84.84


Findings\. FHVHV dominates the TLC ecosystem, accounting for nearly 85% of all trips during the study period\. Yellow taxis contribute roughly 15% of trips, while Green taxis represent less than 1% of total activity\. Based on volume alone, Green appears too small to justify its own standalone mobility series, while the more meaningful modeling decision is whether Yellow and FHVHV should be analyzed together or separately\.

### Temporal Correlation

Before comparing the services visually, let's quantify how closely they move together over time\. If Yellow, Green, and FHVHV exhibit similar day\-to\-day patterns, then a pooled TLC series is likely capturing most of the underlying signal\. If the correlations are substantially lower, then separate series may contain additional information that would be lost through pooling\.

In [32]:

tlc_daily_correlation_df = con.execute(f"""
    SELECT
        date,
        SUM(yellow_trip_count) AS yellow_trip_count,
        SUM(green_trip_count) AS green_trip_count,
        SUM(fhvhv_trip_count) AS fhvhv_trip_count,
        SUM(tlc_trip_count) AS pooled_tlc_trip_count
    FROM read_parquet('{merged_mobility_layer_path}')
    GROUP BY date
    ORDER BY date
""").df()

display(
    tlc_daily_correlation_df[
        [
            "yellow_trip_count",
            "green_trip_count",
            "fhvhv_trip_count",
            "pooled_tlc_trip_count"
        ]
    ].corr()
)

,yellow_trip_count,green_trip_count,fhvhv_trip_count,pooled_tlc_trip_count
yellow_trip_count,1.000000,0.103404,0.585165,0.721954
green_trip_count,0.103404,1.000000,-0.044868,-0.011621
fhvhv_trip_count,0.585165,-0.044868,1.000000,0.983559
pooled_tlc_trip_count,0.721954,-0.011621,0.983559,1.000000


Findings\. This argues against treating all TLC services as interchangeable\. The pooled TLC series is almost perfectly correlated with FHVHV, which makes sense because FHVHV accounts for most trips\. Yellow is only moderately correlated with FHVHV and the pooled series, while Green is basically moving on its own and contributes almost no volume\. This suggests the pooled TLC signal is mostly an FHVHV signal, not a balanced representation of all three services\.

Trip counts tell us whether demand moves together, but mobility systems can still behave differently even when volumes are correlated\. To determine whether Yellow and FHVHV exhibit similar transportation behavior, we'll compare the temporal correlation of trip distance, duration, and speed across the study period\.

In [33]:
tlc_distance_correlation_df = con.execute(f"""
    SELECT
        date,

        AVG(yellow_avg_trip_distance) AS yellow_avg_trip_distance,
        AVG(green_avg_trip_distance) AS green_avg_trip_distance,
        AVG(fhvhv_avg_trip_distance) AS fhvhv_avg_trip_distance,
        AVG(tlc_avg_trip_distance) AS pooled_avg_trip_distance

    FROM read_parquet('{merged_mobility_layer_path}')
    GROUP BY date
    ORDER BY date
""").df()

display(
    tlc_distance_correlation_df[
        [
            "yellow_avg_trip_distance",
            "green_avg_trip_distance",
            "fhvhv_avg_trip_distance",
            "pooled_avg_trip_distance"
        ]
    ].corr()
)

,yellow_avg_trip_distance,green_avg_trip_distance,fhvhv_avg_trip_distance,pooled_avg_trip_distance
yellow_avg_trip_distance,1.000000,0.405560,-0.119288,-0.119273
green_avg_trip_distance,0.405560,1.000000,-0.066892,-0.060388
fhvhv_avg_trip_distance,-0.119288,-0.066892,1.000000,0.992698
pooled_avg_trip_distance,-0.119273,-0.060388,0.992698,1.000000


In [34]:
tlc_duration_correlation_df = con.execute(f"""
    SELECT
        date,

        AVG(yellow_avg_trip_duration) AS yellow_avg_trip_duration,
        AVG(green_avg_trip_duration) AS green_avg_trip_duration,
        AVG(fhvhv_avg_trip_duration) AS fhvhv_avg_trip_duration,
        AVG(tlc_avg_trip_duration) AS pooled_avg_trip_duration

    FROM read_parquet('{merged_mobility_layer_path}')
    GROUP BY date
    ORDER BY date
""").df()

display(
    tlc_duration_correlation_df[
        [
            "yellow_avg_trip_duration",
            "green_avg_trip_duration",
            "fhvhv_avg_trip_duration",
            "pooled_avg_trip_duration"
        ]
    ].corr()
)

,yellow_avg_trip_duration,green_avg_trip_duration,fhvhv_avg_trip_duration,pooled_avg_trip_duration
yellow_avg_trip_duration,1.000000,0.315159,0.597505,0.632605
green_avg_trip_duration,0.315159,1.000000,0.291270,0.313635
fhvhv_avg_trip_duration,0.597505,0.291270,1.000000,0.994617
pooled_avg_trip_duration,0.632605,0.313635,0.994617,1.000000


Findings\. Yellow and FHVHV exhibit very different distance and duration patterns over time\. Correlations between the two services are close to zero for both metrics, suggesting they are not responding to the same underlying mobility dynamics\. The pooled distance and duration series also track FHVHV much more closely than Yellow, reinforcing that the pooled TLC signal is heavily driven by FHVHV activity\.

### Behavioral Comparison

In [35]:

tlc_service_behavior_df = con.execute(f"""
    SELECT
        'Yellow' AS service,
        SUM(yellow_trip_count) AS trip_count,
        ROUND(
            SUM(yellow_avg_trip_distance * yellow_trip_count)
            / NULLIF(SUM(yellow_trip_count), 0),
            2
        ) AS weighted_avg_trip_distance,
        ROUND(
            SUM(yellow_avg_trip_duration * yellow_trip_count)
            / NULLIF(SUM(yellow_trip_count), 0),
            2
        ) AS weighted_avg_trip_duration,
        ROUND(
            SUM(yellow_avg_trip_speed * yellow_trip_count)
            / NULLIF(SUM(yellow_trip_count), 0),
            2
        ) AS weighted_avg_trip_speed
    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'Green' AS service,
        SUM(green_trip_count) AS trip_count,
        ROUND(
            SUM(green_avg_trip_distance * green_trip_count)
            / NULLIF(SUM(green_trip_count), 0),
            2
        ) AS weighted_avg_trip_distance,
        ROUND(
            SUM(green_avg_trip_duration * green_trip_count)
            / NULLIF(SUM(green_trip_count), 0),
            2
        ) AS weighted_avg_trip_duration,
        ROUND(
            SUM(green_avg_trip_speed * green_trip_count)
            / NULLIF(SUM(green_trip_count), 0),
            2
        ) AS weighted_avg_trip_speed
    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'FHVHV' AS service,
        SUM(fhvhv_trip_count) AS trip_count,
        ROUND(
            SUM(fhvhv_avg_trip_distance * fhvhv_trip_count)
            / NULLIF(SUM(fhvhv_trip_count), 0),
            2
        ) AS weighted_avg_trip_distance,
        ROUND(
            SUM(fhvhv_avg_trip_duration * fhvhv_trip_count)
            / NULLIF(SUM(fhvhv_trip_count), 0),
            2
        ) AS weighted_avg_trip_duration,
        ROUND(
            SUM(fhvhv_avg_trip_speed * fhvhv_trip_count)
            / NULLIF(SUM(fhvhv_trip_count), 0),
            2
        ) AS weighted_avg_trip_speed
    FROM read_parquet('{merged_mobility_layer_path}')

    UNION ALL

    SELECT
        'Pooled TLC' AS service,
        SUM(tlc_trip_count) AS trip_count,
        ROUND(
            SUM(tlc_avg_trip_distance * tlc_trip_count)
            / NULLIF(SUM(tlc_trip_count), 0),
            2
        ) AS weighted_avg_trip_distance,
        ROUND(
            SUM(tlc_avg_trip_duration * tlc_trip_count)
            / NULLIF(SUM(tlc_trip_count), 0),
            2
        ) AS weighted_avg_trip_duration,
        ROUND(
            SUM(tlc_avg_trip_speed * tlc_trip_count)
            / NULLIF(SUM(tlc_trip_count), 0),
            2
        ) AS weighted_avg_trip_speed
    FROM read_parquet('{merged_mobility_layer_path}')
""").df()

display(tlc_service_behavior_df)

,service,trip_count,weighted_avg_trip_distance,weighted_avg_trip_duration,weighted_avg_trip_speed
0,Yellow,136986161.0,3.44,1061.22,10.65
1,Green,2091290.0,3.01,1241.83,11.16
2,FHVHV,778288736.0,5.02,1192.51,13.44
3,Pooled TLC,917366187.0,4.78,1173.02,13.02


Findings\. Green looks fundamentally different from the other services, but it also represents less than 1% of trips\. Yellow and FHVHV exhibit similar trip distances, although FHVHV trips tend to be longer and slower\. The pooled TLC metrics closely resemble FHVHV, reflecting FHVHV's dominant share of total trips\. Together with the correlation analysis, these results suggest that Green is unlikely to warrant separate treatment, while Yellow may still contain information that is distinct from FHVHV\.

### Geographic Check

Temporal and behavioral differences tell us whether the services move differently over time, but they do not tell us whether they serve different parts of the city\. To complete the pooling assessment, we'll compare the geographic footprint of Yellow, Green, and FHVHV trips\. If the services exhibit similar spatial distributions, pooling becomes easier to justify\. If they are concentrated in different boroughs, then separate series may preserve meaningful geographic information that would otherwise be lost\.

In [36]:
tlc_borough_distribution_df = con.execute(f"""
    SELECT
        borough,

        SUM(yellow_trip_count) AS yellow_trip_count,
        SUM(green_trip_count) AS green_trip_count,
        SUM(fhvhv_trip_count) AS fhvhv_trip_count,
        SUM(tlc_trip_count) AS pooled_tlc_trip_count,

        ROUND(
            100.0 * SUM(yellow_trip_count)
            / SUM(SUM(yellow_trip_count)) OVER (),
            2
        ) AS pct_yellow,

        ROUND(
            100.0 * SUM(green_trip_count)
            / SUM(SUM(green_trip_count)) OVER (),
            2
        ) AS pct_green,

        ROUND(
            100.0 * SUM(fhvhv_trip_count)
            / SUM(SUM(fhvhv_trip_count)) OVER (),
            2
        ) AS pct_fhvhv

    FROM read_parquet('{merged_mobility_layer_path}')
    GROUP BY borough
    ORDER BY pooled_tlc_trip_count DESC
""").df()

display(tlc_borough_distribution_df)

,borough,yellow_trip_count,green_trip_count,fhvhv_trip_count,pooled_tlc_trip_count,pct_yellow,pct_green,pct_fhvhv
0,Manhattan,119901024.0,1241470.0,299224144.0,420366638.0,87.53,59.36,38.45
1,Brooklyn,3055423.0,302334.0,206060180.0,209417937.0,2.23,14.46,26.48
2,Queens,12774958.0,508201.0,164158412.0,177441571.0,9.33,24.30,21.09
3,Bronx,665676.0,36598.0,97174914.0,97877188.0,0.49,1.75,12.49
4,Staten Island,9186.0,166.0,11637220.0,11646572.0,0.01,0.01,1.50
5,Unknown,577697.0,2498.0,33807.0,614002.0,0.42,0.12,0.00
6,EWR,2197.0,23.0,59.0,2279.0,0.00,0.00,0.00


Findings\. The geographic patterns differ substantially across services\. Yellow trips are overwhelmingly concentrated in Manhattan, with nearly 90% of all activity originating there\. FHVHV activity is much more geographically distributed, with meaningful volumes across Brooklyn, Queens, and the Bronx\. Green taxis also exhibit a different footprint than Yellow, although their overall volume is very small\. These results reinforce the earlier temporal and behavioral analyses, suggesting that Yellow and FHVHV capture different mobility patterns and should not be treated as interchangeable signals\.

### Section Summary and Recommendation

The results point to a pretty clear answer: 

- Green contributes almost no activity and doesn't appear large enough to justify its own mobility series\. At the other extreme, the pooled TLC metrics are largely an FHVHV signal because FHVHV accounts for roughly 85% of all trips\. 

- Yellow and FHVHV, however, consistently behaved differently throughout the analysis\. They exhibited only moderate trip\-count correlation, very different distance and duration patterns, and noticeably different geographic footprints\. 

- Yellow activity remains heavily concentrated in Manhattan, while FHVHV activity is much more broadly distributed across the city\. 

Based on these findings, we'll combine Yellow and Green into a single Taxi mobility series and retain FHVHV as a separate modality\. This preserves the most meaningful behavioral differences while avoiding unnecessary feature duplication and complexity in the final mobility panel\.

## 1\.3\.1\.8 Write Final Outputs

We've now made the major design decisions for the mobility panel, including how to handle missing values and how to represent TLC activity\. The final step is to apply those decisions and write the finished analytical layer\. This includes combining Yellow and Green into a single Taxi mobility series, retaining FHVHV as a separate mobility modality, and removing the intermediate TLC representations used during validation\. The resulting dataset will contain a cleaner, more interpretable set of mobility features while avoiding redundant pooled, Yellow\-only, and Green\-only measures\. This streamlined mobility panel will serve as the foundation for feature engineering, dimensionality reduction, anomaly detection, and congestion\-pricing analysis\.

### Reconstruct Taxi Destination Diversity

Because Yellow and Green are being combined into a single Taxi modality, simply summing their distinct dropoff zone counts would overstate destination diversity whenever both services travel to the same locations\. To avoid this double\-counting, we'll recalculate Taxi destination diversity directly from the underlying TLC mobility records by combining Yellow and Green trips and counting unique dropoff zones at the Taxi Zone × Date × Temporal Bucket level\. This produces a more accurate measure of the geographic breadth of Taxi activity before constructing the final audit and analytical datasets\. Since this requires a distinct count over TLC records, we’ll process it quarter by quarter to keep memory usage under control and make progress easier to monitor\.

In [37]:
# ---------------------------------------------------------------------
# Build Taxi distinct dropoff counts in quarter chunks
# ---------------------------------------------------------------------

# Yellow and Green Taxi are combined into one Taxi modality later in this notebook.
# Distinct destination counts cannot be summed across services because both services
# may visit the same dropoff zone in the same Taxi Zone x Date x Temporal Bucket.
quarter_windows = [
    ("2023_q1", "2023-01-01", "2023-04-01"),
    ("2023_q2", "2023-04-01", "2023-07-01"),
    ("2023_q3", "2023-07-01", "2023-10-01"),
    ("2023_q4", "2023-10-01", "2024-01-01"),
    ("2024_q1", "2024-01-01", "2024-04-01"),
    ("2024_q2", "2024-04-01", "2024-07-01"),
    ("2024_q3", "2024-07-01", "2024-10-01"),
    ("2024_q4", "2024-10-01", "2025-01-01"),
    ("2025_q1", "2025-01-01", "2025-04-01"),
    ("2025_q2", "2025-04-01", "2025-07-01"),
    ("2025_q3", "2025-07-01", "2025-10-01"),
    ("2025_q4", "2025-10-01", "2026-01-01"),
    ("2026_q1", "2026-01-01", "2026-04-01"),
]

expected_chunk_paths = [
    taxi_distinct_dropoff_chunk_dir / f"taxi_distinct_dropoff_{quarter_label}.parquet"
    for quarter_label, _, _ in quarter_windows
]

all_chunks_exist = all(path.exists() for path in expected_chunk_paths)
manifest_exists = taxi_distinct_dropoff_manifest_path.exists()

should_build_chunks = (
    REBUILD_TAXI_DISTINCT_DROPOFF_CHUNKS
    or not all_chunks_exist
    or not manifest_exists
)

if REBUILD_TAXI_DISTINCT_DROPOFF_CHUNKS and taxi_distinct_dropoff_chunk_dir.exists():
    shutil.rmtree(taxi_distinct_dropoff_chunk_dir)

taxi_distinct_dropoff_chunk_dir.mkdir(
    parents=True,
    exist_ok=True
)

if should_build_chunks:

    chunk_manifest_records = []

    for quarter_label, start_date, end_date in quarter_windows:

        output_path = taxi_distinct_dropoff_chunk_dir / f"taxi_distinct_dropoff_{quarter_label}.parquet"

        if output_path.exists() and not REBUILD_TAXI_DISTINCT_DROPOFF_CHUNKS:
            print(f"Using existing Taxi distinct dropoff chunk: {output_path}")

            chunk_manifest_records.append({
                "quarter": quarter_label,
                "start_date": start_date,
                "end_date": end_date,
                "path": str(output_path),
                "status": "existing"
            })

            continue

        print(f"Building Taxi distinct dropoff chunk: {quarter_label}")
        start_time = time.time()

        con = duckdb.connect()
        con.execute("PRAGMA threads=1")
        con.execute("PRAGMA memory_limit='3GB'")

        # Recompute the distinct dropoff-zone count from the underlying Yellow/Green records.
        # Quarter chunks keep the distinct-count query memory-safe and restartable.
        con.execute(f"""
            COPY (
                SELECT
                    pickup_zone_id AS taxi_zone_id,
                    date,
                    temporal_bucket,
                    COUNT(DISTINCT dropoff_zone_id) AS taxi_distinct_dropoff_zone_count
                FROM read_parquet('{tlc_dropoff_source_glob}')
                WHERE
                    dataset_type IN ('yellow', 'green')
                    AND pickup_zone_id IS NOT NULL
                    AND dropoff_zone_id IS NOT NULL
                    AND date >= DATE '{start_date}'
                    AND date < DATE '{end_date}'
                GROUP BY
                    pickup_zone_id,
                    date,
                    temporal_bucket
            )
            TO '{output_path}'
            (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        con.close()
        del con
        gc.collect()

        elapsed_minutes = (time.time() - start_time) / 60

        chunk_manifest_records.append({
            "quarter": quarter_label,
            "start_date": start_date,
            "end_date": end_date,
            "path": str(output_path),
            "status": "built",
            "elapsed_minutes": round(elapsed_minutes, 2)
        })

        print(f"Finished {quarter_label} in {elapsed_minutes:.2f} minutes")

    manifest = {
        "description": "Quarterly Taxi Yellow + Green distinct dropoff-zone chunks",
        "source_glob": tlc_dropoff_source_glob,
        "chunk_dir": str(taxi_distinct_dropoff_chunk_dir),
        "chunk_count": len(expected_chunk_paths),
        "chunks": chunk_manifest_records
    }

    with open(taxi_distinct_dropoff_manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"Wrote manifest: {taxi_distinct_dropoff_manifest_path}")

else:
    print("Using existing Taxi distinct dropoff chunks and manifest.")
    print(f"Chunk directory: {taxi_distinct_dropoff_chunk_dir}")
    print(f"Manifest: {taxi_distinct_dropoff_manifest_path}")

Using existing Taxi distinct dropoff chunks and manifest.
Chunk directory: pipeline_data/1.3.1.intermediate_tables/taxi_distinct_dropoff_chunks
Manifest: pipeline_data/1.3.1.intermediate_tables/taxi_distinct_dropoff_chunks_manifest.json


In [38]:
taxi_distinct_dropoff_chunk_glob = str(
    taxi_distinct_dropoff_chunk_dir /
    "*.parquet"
)

con = duckdb.connect()
con.execute("PRAGMA threads=1")
con.execute("PRAGMA memory_limit='3GB'")

taxi_distinct_dropoff_df = con.execute(f"""
    SELECT *
    FROM read_parquet('{taxi_distinct_dropoff_chunk_glob}')
""").df()

con.close()
del con
gc.collect()

display(taxi_distinct_dropoff_df.head())

print(
    f"Taxi distinct dropoff rows: "
    f"{len(taxi_distinct_dropoff_df):,}"
)

,taxi_zone_id,date,temporal_bucket,taxi_distinct_dropoff_zone_count
0,129,2023-01-01,weekend_overnight,26
1,134,2023-01-01,weekend_overnight,2
2,179,2023-01-01,weekend_overnight,13
3,95,2023-01-01,weekend_overnight,7
4,260,2023-01-01,weekend_overnight,14


Taxi distinct dropoff rows: 1,125,143


Findings\. Successfully reconstructed Taxi destination\-diversity metrics directly from the underlying Yellow and Green trip records\. This approach avoids double\-counting destinations and produces 1\.14M valid Taxi Zone × Date × Temporal Bucket observations for inclusion in the final mobility panels\.

### Create Audit\-Only Mobility Panel

The Audit\-Only Mobility Panel applies the final TLC representation while preserving supporting QA, coverage, variability, and diagnostic fields\. This dataset serves as the project's authoritative validation layer and can be used to investigate anomalies, verify transformations, and troubleshoot future issues without requiring regeneration of intermediate datasets\.

In [39]:
# ---------------------------------------------------------------------
# Create audit-only mobility panel with final Taxi/FHVHV representation
# ---------------------------------------------------------------------

con = duckdb.connect()

# The audit panel keeps QA and diagnostic fields while applying the final TLC design:
# Yellow + Green become Taxi, and FHVHV remains a separate mobility modality.
audit_only_mobility_panel_df = con.execute(f"""
    SELECT

        -- Core dimensions
        base.taxi_zone_id,
        base.zone,
        base.borough,
        base.canonical_location_id,
        base.date,
        base.year,
        base.month,
        base.day_of_week,
        base.temporal_bucket,
        base.pre_post_cp,

        -- Traffic
        base.traffic_volume,
        base.traffic_observation_count,
        base.traffic_distinct_segment_count,

        -- Bus
        base.avg_bus_speed,
        base.avg_bus_travel_time,
        base.bus_speed_stddev,
        base.bus_travel_time_stddev,
        base.bus_trip_count,
        base.bus_segment_observation_count,
        base.bus_assignment_method_count,

        -- Subway
        base.subway_ridership,
        base.subway_transfers,
        base.subway_observation_count,
        base.subway_station_complex_count,

        -- Taxi (Yellow + Green)
        base.yellow_trip_count + base.green_trip_count AS taxi_trip_count,

        -- Weighted averages preserve trip-volume weighting when combining Yellow and Green.
        (
            (base.yellow_avg_trip_distance * base.yellow_trip_count)
            +
            (base.green_avg_trip_distance * base.green_trip_count)
        )
        /
        NULLIF(base.yellow_trip_count + base.green_trip_count, 0)
        AS taxi_avg_trip_distance,

        (
            (base.yellow_avg_trip_duration * base.yellow_trip_count)
            +
            (base.green_avg_trip_duration * base.green_trip_count)
        )
        /
        NULLIF(base.yellow_trip_count + base.green_trip_count, 0)
        AS taxi_avg_trip_duration,

        (
            (base.yellow_avg_trip_speed * base.yellow_trip_count)
            +
            (base.green_avg_trip_speed * base.green_trip_count)
        )
        /
        NULLIF(base.yellow_trip_count + base.green_trip_count, 0)
        AS taxi_avg_trip_speed,

        taxi_dropoffs.taxi_distinct_dropoff_zone_count,

        (
            base.yellow_observation_count
            +
            base.green_observation_count
        ) AS taxi_observation_count,

        -- FHVHV
        base.fhvhv_trip_count,
        base.fhvhv_avg_trip_distance,
        base.fhvhv_trip_distance_stddev,
        base.fhvhv_avg_trip_duration,
        base.fhvhv_trip_duration_stddev,
        base.fhvhv_avg_trip_speed,
        base.fhvhv_trip_speed_stddev,
        base.fhvhv_distinct_dropoff_zone_count,
        base.fhvhv_observation_count

    FROM read_parquet('{fixed_merged_mobility_layer_path}') base

    LEFT JOIN taxi_distinct_dropoff_df taxi_dropoffs
        ON base.taxi_zone_id = taxi_dropoffs.taxi_zone_id
        AND base.date = taxi_dropoffs.date
        AND base.temporal_bucket = taxi_dropoffs.temporal_bucket
""").df()

display(audit_only_mobility_panel_df.head())

print(
    f"Audit-only mobility panel rows: "
    f"{len(audit_only_mobility_panel_df):,}"
)

,taxi_zone_id,zone,borough,canonical_location_id,date,year,month,day_of_week,temporal_bucket,pre_post_cp,...,taxi_observation_count,fhvhv_trip_count,fhvhv_avg_trip_distance,fhvhv_trip_distance_stddev,fhvhv_avg_trip_duration,fhvhv_trip_duration_stddev,fhvhv_avg_trip_speed,fhvhv_trip_speed_stddev,fhvhv_distinct_dropoff_zone_count,fhvhv_observation_count
0,42,Central Harlem North,Manhattan,42,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,15.0,977.0,3.960017,3.923033,1143.654043,764.904768,11.136213,4.254374,131,297.0
1,25,Boerum Hill,Brooklyn,25,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,11.0,660.0,3.947827,4.020738,1362.762121,903.803608,9.410097,3.593390,120,255.0
2,68,East Chelsea,Manhattan,68,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,225.0,1768.0,4.429926,4.865223,1464.921946,842.716689,9.303354,5.399891,133,348.0
3,112,Greenpoint,Brooklyn,112,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,3.0,856.0,3.723801,3.201157,1245.099299,867.668406,10.409876,2.660512,116,263.0
4,130,Jamaica,Queens,130,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,14.0,1050.0,3.936255,3.366515,1069.911429,592.206361,12.057951,4.532143,98,229.0


Audit-only mobility panel rows: 1,559,590


### Create Analysis\-Ready Mobility Panel

With the audit layer preserved, we can create a streamlined analytical dataset focused on mobility signals rather than data quality diagnostics\. This version removes supporting QA and coverage fields, leaving a cleaner set of features for feature engineering, dimensionality reduction, anomaly detection, and congestion\-pricing analysis\.

In [40]:
con = duckdb.connect()
analysis_ready_mobility_panel_df = con.execute("""
    SELECT

        -- Core dimensions
        taxi_zone_id,
        zone,
        borough,
        canonical_location_id,
        date,
        year,
        month,
        day_of_week,
        temporal_bucket,
        pre_post_cp,

        -- Traffic
        traffic_volume,

        -- Bus
        bus_trip_count,
        avg_bus_speed,
        avg_bus_travel_time,

        -- Subway
        subway_ridership,
        subway_transfers,

        -- Taxi
        taxi_trip_count,
        taxi_avg_trip_distance,
        taxi_avg_trip_duration,
        taxi_avg_trip_speed,
        taxi_distinct_dropoff_zone_count,

        -- FHVHV
        fhvhv_trip_count,
        fhvhv_avg_trip_distance,
        fhvhv_avg_trip_duration,
        fhvhv_avg_trip_speed,
        fhvhv_distinct_dropoff_zone_count

    FROM audit_only_mobility_panel_df
""").df()

display(analysis_ready_mobility_panel_df.head())

print(
    f"Analysis-ready mobility panel rows: "
    f"{len(analysis_ready_mobility_panel_df):,}"
)

,taxi_zone_id,zone,borough,canonical_location_id,date,year,month,day_of_week,temporal_bucket,pre_post_cp,...,taxi_trip_count,taxi_avg_trip_distance,taxi_avg_trip_duration,taxi_avg_trip_speed,taxi_distinct_dropoff_zone_count,fhvhv_trip_count,fhvhv_avg_trip_distance,fhvhv_avg_trip_duration,fhvhv_avg_trip_speed,fhvhv_distinct_dropoff_zone_count
0,42,Central Harlem North,Manhattan,42,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,16.0,2.475625,1230.125000,7.826937,11,977.0,3.960017,1143.654043,11.136213,131
1,25,Boerum Hill,Brooklyn,25,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,11.0,4.268182,1431.727273,9.331850,9,660.0,3.947827,1362.762121,9.410097,120
2,68,East Chelsea,Manhattan,68,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,877.0,2.402965,1114.020525,7.405056,86,1768.0,4.429926,1464.921946,9.303354,133
3,112,Greenpoint,Brooklyn,112,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,3.0,6.363333,1391.000000,14.991969,3,856.0,3.723801,1245.099299,10.409876,116
4,130,Jamaica,Queens,130,2023-10-10,2023,10,Tuesday,weekday_pm_peak,pre_cp,...,16.0,3.606250,1274.187500,11.532444,12,1050.0,3.936255,1069.911429,12.057951,98


Analysis-ready mobility panel rows: 1,559,590


### Write to File

In [41]:
audit_only_output_path = MERGED_FINAL_DIR / "audit_only_mobility_panel.parquet"
analysis_ready_output_path = MERGED_FINAL_DIR / "analysis_ready_mobility_panel.parquet"

audit_only_mobility_panel_df.to_parquet(
    audit_only_output_path,
    index=False
)

analysis_ready_mobility_panel_df.to_parquet(
    analysis_ready_output_path,
    index=False
)

print(f"Audit output: {audit_only_output_path}")
print(f"Analysis output: {analysis_ready_output_path}")

Audit output: pipeline_data/1.3.1.final_tables/audit_only_mobility_panel.parquet
Analysis output: pipeline_data/1.3.1.final_tables/analysis_ready_mobility_panel.parquet


Copy the Bridge & Tunnel file forward

In [42]:
bridge_tunnel_output_path = MERGED_FINAL_DIR / "bridge_tunnel_mobility_panel.parquet"

con.execute(f"""
    COPY (
        SELECT *
        FROM read_parquet('{bridge_tunnel_path}')
    )
    TO '{bridge_tunnel_output_path}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Bridge & Tunnel output: {bridge_tunnel_output_path}")

Bridge & Tunnel output: pipeline_data/1.3.1.final_tables/bridge_tunnel_mobility_panel.parquet


Copy the Spatial reference file over for convenience as well\.

In [43]:
# ---------------------------------------------------------------------
# Carry forward harmonized Taxi Zone reference layer for downstream use
# ---------------------------------------------------------------------

taxi_zones_output_path = (
    MERGED_FINAL_DIR / "nyc_taxi_zones_harmonized.parquet"
)

taxi_zones_df = pd.read_parquet(taxi_zones_path)

taxi_zones_df["geometry"] = taxi_zones_df["geometry"].apply(
    lambda geom: wkb.loads(geom) if pd.notna(geom) else None
)

taxi_zones_gdf = gpd.GeoDataFrame(
    taxi_zones_df,
    geometry="geometry",
    crs="EPSG:4326",
)

taxi_zones_projected_gdf = taxi_zones_gdf.to_crs("EPSG:2263").copy()

taxi_zones_projected_gdf["centroid_x"] = (
    taxi_zones_projected_gdf.geometry.centroid.x
)

taxi_zones_projected_gdf["centroid_y"] = (
    taxi_zones_projected_gdf.geometry.centroid.y
)

taxi_zones_output_df = pd.DataFrame(
    taxi_zones_projected_gdf.to_crs("EPSG:4326")
)

taxi_zones_output_df["geometry"] = (
    taxi_zones_output_df["geometry"]
    .apply(lambda geom: geom.wkb if geom is not None else None)
)

taxi_zones_output_df.to_parquet(
    taxi_zones_output_path,
    index=False,
)

print(f"Taxi Zones output: {taxi_zones_output_path}")

Taxi Zones output: pipeline_data/1.3.1.final_tables/nyc_taxi_zones_harmonized.parquet


Let's do one final QA

In [44]:
pd.DataFrame({
    "dataset": [
        "Audit Only",
        "Analysis Ready"
    ],
    "rows": [
        len(audit_only_mobility_panel_df),
        len(analysis_ready_mobility_panel_df)
    ],
    "columns": [
        len(audit_only_mobility_panel_df.columns),
        len(analysis_ready_mobility_panel_df.columns)
    ]
})

,dataset,rows,columns
0,Audit Only,1559590,39
1,Analysis Ready,1559590,26


## Close

This notebook transformed the harmonized mobility datasets into a unified multimodal mobility panel suitable for downstream analysis\. We constructed a dense Taxi Zone × Date × Temporal Bucket panel, integrated Traffic, Bus, Subway, and TLC mobility measures, investigated modality\-specific missingness, established final NaN handling policies, and evaluated alternative TLC representations\. The final mobility design treats Taxi \(Yellow \+ Green\) and FHVHV as separate mobility modalities, preserving meaningful behavioral differences while reducing feature duplication\. The notebook concludes by generating both an Audit\-Only Mobility Panel for validation and troubleshooting and an Analysis\-Ready Mobility Panel for downstream modeling\.

### Outputs for Downstream Analysis

The following datasets were generated in this notebook:

Analysis\-Ready Mobility Panel \(Primary Dataset\)

- pipeline\_data/1\.3\.1\.final\_tables/analysis\_ready\_mobility\_panel\.parquet

- Use for: EDA, feature engineering, dimensionality reduction, anomaly detection, congestion\-pricing analysis, and all downstream modeling activities\.

Audit\-Only Mobility Panel \(Validation & Troubleshooting\)

- pipeline\_data/1\.3\.1\.final\_tables/audit\_only\_mobility\_panel\.parquet

- Use for: QA, validation, troubleshooting, missing\-data investigations, metric reconciliation, and methodological review\.

Bridge & Tunnel Mobility Dataset \(Facility\-Level Context\)

- pipeline\_data/1\.3\.1\.final\_tables/bridge\_tunnel\_mobility\_panel\.parquet

- Use for: contextual mobility analysis, system\-level validation, and potential anomaly corroboration\. This dataset remains separate from the Taxi Zone mobility panel\.

Taxi Zone Reference Layer \(Canonical Geography\)

- pipeline\_data/1\.3\.1\.final\_tables/nyc\_taxi\_zones\_harmonized\.parquet

- Use for: spatial feature engineering, geospatial visualization, neighborhood spillover analysis, borough\-level aggregation, congestion\-pricing geography classification, CBD proximity calculations, and all downstream spatial joins\. This dataset serves as the project's canonical geographic reference layer and should be used whenever Taxi Zone attributes, boundaries, or spatial relationships are required\.

### Looking Ahead

The next notebooks \(1\.4\.x Exploratory Data Analysis and 1\.5x Feature Engineering\) should primarily load:

- pipeline\_data/1\.3\.1\.final\_tables/analysis\_ready\_mobility\_panel\.parquet

- pipeline\_data/1\.3\.1\.final\_tables/nyc\_taxi\_zones\_harmonized\.parquet

- pipeline\_data/1\.3\.1\.final\_tables/bridge\_tunnel\_mobility\_panel\.parquet \(optional contextual dataset\)

The Audit\-Only Mobility Panel should only be consulted when validating results or investigating unexpected behavior discovered during EDA or feature engineering\. From this point forward, all analytical work should be based on the Analysis\-Ready Mobility Panel\.

In [45]:
# ---------------------------------------------------------------------
# Archived TLC repair QA: metric quality in merged mobility layer
# ---------------------------------------------------------------------
# This was used during the TLC cleanup repair pass. It is intentionally
# commented out so a full Run Notebook does not execute debug QA after
# the notebook Close section.
#
# To rerun manually, uncomment this cell after the 1.3.1 analysis-ready
# mobility panel has been written.
#
# from pathlib import Path
# import duckdb
# import pandas as pd
#
# PIPELINE_DATA_DIR = Path("pipeline_data")
# merged_panel_path = PIPELINE_DATA_DIR / "1.3.1.final_tables" / "analysis_ready_mobility_panel.parquet"
#
# MIN_VALID_TLC_TRIP_DURATION_SECONDS = 60
# MAX_VALID_TLC_TRIP_DURATION_SECONDS = 86_400
# MIN_VALID_TLC_TRIP_DISTANCE = 0
# MAX_VALID_TLC_TRIP_DISTANCE = 100
# MAX_VALID_TLC_TRIP_SPEED_MPH = 100
#
# tlc_metric_quality_query = f"""
#     SELECT
#         COUNT(*) AS row_count,
#
#         SUM(CASE WHEN taxi_trip_count > 0 AND taxi_avg_trip_duration < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS short_taxi_duration_rows,
#         SUM(CASE WHEN taxi_trip_count > 0 AND taxi_avg_trip_duration > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS excessive_taxi_duration_rows,
#         SUM(CASE WHEN taxi_trip_count > 0 AND taxi_avg_trip_distance < {MIN_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS negative_taxi_distance_rows,
#         SUM(CASE WHEN taxi_trip_count > 0 AND taxi_avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS excessive_taxi_distance_rows,
#         SUM(CASE WHEN taxi_trip_count > 0 AND taxi_avg_trip_speed > {MAX_VALID_TLC_TRIP_SPEED_MPH} THEN 1 ELSE 0 END)
#             AS excessive_taxi_speed_rows,
#
#         SUM(CASE WHEN fhvhv_trip_count > 0 AND fhvhv_avg_trip_duration < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS short_fhvhv_duration_rows,
#         SUM(CASE WHEN fhvhv_trip_count > 0 AND fhvhv_avg_trip_duration > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS excessive_fhvhv_duration_rows,
#         SUM(CASE WHEN fhvhv_trip_count > 0 AND fhvhv_avg_trip_distance < {MIN_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS negative_fhvhv_distance_rows,
#         SUM(CASE WHEN fhvhv_trip_count > 0 AND fhvhv_avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS excessive_fhvhv_distance_rows,
#         SUM(CASE WHEN fhvhv_trip_count > 0 AND fhvhv_avg_trip_speed > {MAX_VALID_TLC_TRIP_SPEED_MPH} THEN 1 ELSE 0 END)
#             AS excessive_fhvhv_speed_rows,
#
#         MAX(CASE WHEN taxi_trip_count > 0 THEN taxi_avg_trip_speed ELSE NULL END)
#             AS max_active_taxi_avg_trip_speed,
#         MAX(CASE WHEN fhvhv_trip_count > 0 THEN fhvhv_avg_trip_speed ELSE NULL END)
#             AS max_active_fhvhv_avg_trip_speed
#
#     FROM read_parquet('{merged_panel_path.as_posix()}')
# """
#
# tlc_metric_quality_validation_df = duckdb.sql(tlc_metric_quality_query).df()
#
# float_cols = tlc_metric_quality_validation_df.select_dtypes(include=["float"]).columns
# tlc_metric_quality_validation_df[float_cols] = tlc_metric_quality_validation_df[float_cols].round(3)
#
# display(tlc_metric_quality_validation_df)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>